In [ ]:
# ==============================================================================
# SPATIOTEMPORAL PU LEARNING WORKFLOW FOR COPPER PROSPECTIVITY
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Import PU learning modules
import sys
# sys.path.append('<PATH_TO>/pyDTDM')
from pyDTDM.pu_learning import PUBaggingClassifier
from pyDTDM.pu_diagnostics import PUFeatureDiagnostics

# Simple PU data validation function
def validate_pu_data(y, verbose=True):
    """Validate PU learning dataset composition"""
    n_positive = np.sum(y == 1)
    n_unlabeled = np.sum(y == 0)
    n_total = len(y)
    positive_rate = n_positive / n_total
    
    diagnostics = {
        'n_positive': n_positive,
        'n_unlabeled': n_unlabeled,
        'n_total': n_total,
        'positive_rate': positive_rate,
        'is_valid': n_positive > 0 and n_unlabeled > 0
    }
    
    if verbose:
        print(f"\nPU Data Validation:")
        print(f"  ✓ Valid PU dataset" if diagnostics['is_valid'] else "  ✗ Invalid PU dataset")
        print(f"  Class imbalance ratio: {n_unlabeled/n_positive:.1f}:1 (unlabeled:positive)")
        if positive_rate < 0.01:
            print(f"  ⚠ Very low positive rate ({positive_rate*100:.2f}%) - typical for PU learning")
        elif positive_rate > 0.5:
            print(f"  ⚠ High positive rate ({positive_rate*100:.2f}%) - unusual for PU learning")
    
    return diagnostics

# Set random state for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*80)
print("SPATIOTEMPORAL PU LEARNING WORKFLOW - COPPER PROSPECTIVITY")
print("="*80)
print(f"Random State: {RANDOM_STATE}")
print(f"Libraries loaded successfully")

In [ ]:
# =============================================================================
# GLOBAL PLOTTING STYLE CONFIGURATION
# =============================================================================
# Run this cell once at the start to ensure consistent publication-quality fonts

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Apply publication quality style
plt.style.use('seaborn-v0_8-paper')

# Global font configuration
plt.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.titlesize': 13,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'axes.linewidth': 1.2,
    'grid.linewidth': 0.5,
    'lines.linewidth': 1.5,
})

print("✓ Publication style configuration applied")
print(f"  - Style: seaborn-v0_8-paper")
print(f"  - Font family: {plt.rcParams['font.family']}")
print(f"  - Font size: {plt.rcParams['font.size']}")
print("\nAll subsequent plots will use these consistent settings.")

# Spatiotemporal PU Learning Workflow for Copper Prospectivity
## Complete Workflow with Feature Grouping and Comprehensive Visualizations

---

### Overview

This notebook implements a **publication-ready** Positive-Unlabeled (PU) learning workflow for spatiotemporal copper prospectivity mapping. The workflow properly handles:

- **Positive samples** = Known copper deposits (reliable labels)
- **Unlabeled samples** = Background locations (NOT negatives - may contain undiscovered deposits)
- **Temporal dimension** = Time evolution of tectonic/geological features

---

### Key Features

1. **Spatiotemporal Feature Engineering**
   - Temporal features (age, time-dependent tectonic variables)
   - Spatial coordinates (present and paleo-positions)
   - Plate tectonic variables (convergence rates, slab geometry)
   - Geology-aware feature grouping by category

2. **PU-Bagging Classifier**
   - All positive samples in every bag
   - Bootstrap sampling of unlabeled samples only
   - XGBoost base estimator with regularization
   - Uncertainty quantification via prediction variance

3. **Comprehensive Evaluation Metrics**
   - **Success-rate curves** (cumulative deposits vs area explored)
   - **Lift curves** (model performance vs random baseline)
   - **Recall@K** (deposits captured in top-K% area)
   - **Enrichment factor** (improvement over random expectation)
   - **Deposit ranking analysis** (where do known deposits appear in predictions)

4. **Feature Importance with Grouping**
   - Individual feature importance rankings
   - Group-level importance (temporal, spatial, tectonic, etc.)
   - Stability analysis across bags
   - Geological interpretability

5. **High-Prospectivity Target Identification**
   - Identify unlabeled samples with deposit-like scores
   - Rank potential discovery targets
   - Export for follow-up exploration

---

### Workflow Steps

1. **Load Data** - Import spatiotemporal training dataset
2. **EDA** - Explore temporal, spatial, and label distributions
3. **Feature Prep** - Handle missing values, remove zero-variance features
4. **Feature Grouping** (Optional) - Organize features by geological category
5. **Train-Test Split** - Stratified split with optional downsampling
6. **Train PU Model** - PU Bagging with tonnage-weighted samples
7. **Predict** - Generate predictions with uncertainty estimates
8. **PU-Aware Evaluation** - Recall@K, Enrichment, Success-rate curves
9. **Visualizations** - 4-panel success-rate curve plots
10. **Feature Importance** - Individual and group-level analysis
11. **Target Identification** - Find high-prospectivity unlabeled locations

---

### Expected Outputs

**CSV Files:**
- `spatiotemporal_PU_metrics.csv` - Recall@K and Enrichment values
- `spatiotemporal_success_rate_curve_data.csv` - Success rate at key percentiles
- `spatiotemporal_deposit_rankings.csv` - Rank of each known deposit
- `spatiotemporal_feature_importance.csv` - Feature rankings
- `spatiotemporal_group_importance.csv` - Group-level importance
- `spatiotemporal_high_prospectivity_targets.csv` - Potential discoveries

**Visualizations:**
- `spatiotemporal_EDA.png` - 4-panel exploratory analysis
- `spatiotemporal_feature_groups.png` - Feature distribution by group
- `spatiotemporal_predictions.png` - Prediction distributions
- `spatiotemporal_prospectivity_evaluation.png` - **4-panel success-rate curves**
- `spatiotemporal_feature_importance.png` - Feature and group importance
- `spatiotemporal_potential_discoveries.png` - High-prospectivity analysis

---

### Key Differences from Standard Classification

❌ **Don't**: Treat unlabeled as negative (causes bias)  
❌ **Don't**: Use Precision (penalizes high-scoring unlabeled samples)  
✅ **Do**: Focus evaluation on known deposit rankings  
✅ **Do**: Use Recall@K and Enrichment metrics  
✅ **Do**: Identify high-prospectivity unlabeled as potential discoveries

---

### Citation

If using this workflow, please cite:
- pyDTDM PU-Learning implementation
- Elkan & Noto (2008) - PU learning foundation
- Breiman (1996) - Bagging concept

---

**Status**: Production-ready for scientific publication  
**Suitable for**: Ore Geology Reviews, Economic Geology, Geoscience Frontiers  
**Author**: pyDTDM Development Team  
**Date**: February 2026

---

In [ ]:
# ==============================================================================
# STEP 1: LOAD SPATIOTEMPORAL TRAINING DATA
# ==============================================================================
print("\n" + "="*80)
print("STEP 1: Load Spatiotemporal Training Data")
print("="*80)

# Load the dataset
data_path = "<DATA_ROOT>/Paper/Zenodo_DataBundle/training_data/spatiotemporal_training_data_Alfonso2024.csv"
training_df = pd.read_csv(data_path)
# training_df['total_precipitation (km)'] = float(training_df['total_precipitation (km)']) / float(training_df['age (Ma)'])  # Convert to km

# training_df['total_precipitation_per_Ma (km/Ma)'] = (
#     training_df['total_precipitation (km)'] / training_df['age (Ma)']
# )

# mask = training_df['age (Ma)'] >= 0

# training_df.loc[mask, 'total_precipitation_per_Ma (km/Ma)'] = (
#     training_df.loc[mask, 'total_precipitation (km)'] /
#     training_df.loc[mask, 'age (Ma)']
# )

# plt.hist(training_df['total_precipitation_per_Ma (km/Ma)'], bins=50)

print(f"Dataset loaded successfully")
print(f"  Shape: {training_df.shape}")
print(f"  Total samples: {len(training_df):,}")
print(f"  Total features: {len(training_df.columns)}")

# Display column info
print(f"\nDataset columns ({len(training_df.columns)} total):")
print(f"  Spatial: present_lon, present_lat, lon, lat")
print(f"  Temporal: age (Ma)")
print(f"  Target: label_binary")
print(f"  Weight: tonnage_mt")
print(f"  Features: {len(training_df.columns) - 7} geological/tectonic features")

# Check data types
print(f"\nData types:")
print(training_df.dtypes.value_counts())

# Memory usage
print(f"\nMemory usage: {training_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Load the grid data
grid_data_path = "<DATA_ROOT>/CopperLithium/NorthAmerica/grid/grid_data_170to0.csv"
print(f"\nLoading grid data from: {grid_data_path}")
grid_df = pd.read_csv(grid_data_path)
print(f"Grid dataset loaded successfully") 
print(f"  columns: {grid_df.columns}")

In [ ]:
print(f"shape: {grid_df.shape}")

In [ ]:
columns_to_use=['present_lon', 'present_lat',
       'trench_velocity_obliquity (degrees)', 
       'seafloor_age (Ma)',
       'distance_to_trench_edge (degrees)', 
       'trench_velocity_orthogonal (cm/yr)',
       'subducted_carbonates_volume (m)', 
       'trench_velocity (cm/yr)',
       'subducted_water_volume (m)',
       'subducting_plate_absolute_obliquity (degrees)',
       'convergence_obliquity (degrees)', 
       'co2_volume (m^3/m^2)',
       'subducted_sediment_volume (m)', 
       'convergence_rate (cm/yr)',
       'sediment_thickness (m)',  
       'carbonate_thickness (m)',
       'convergence_rate_orthogonal (cm/yr)',
       'subducting_plate_absolute_velocity_orthogonal (cm/yr)',
       'subducting_plate_absolute_velocity_parallel (cm/yr)',
       'subducted_plate_volume (m)',
         'convergence_rate_parallel (cm/yr)',
       'subducting_plate_absolute_velocity (cm/yr)',
       'seafloor_spreading_rate (km/Myr)',
       'trench_velocity_parallel (cm/yr)',
         'distance_to_trench (km)',
       'water_thickness (m)', 
       'slab_flux (m^2/yr)',
       'crustal_thickness_mean (m)',
        'crustal_thickness_min (m)',
       'crustal_thickness_max (m)', 'crustal_thickness_median (m)',
       'crustal_thickness_std (m)',
       'crustal_thickness_range (m)', 
       'slab_dip (degrees)', 
       'arc_trench_distance (km)',
      #  'total_precipitation (km)', 
       'weights', 'label_binary',"tonnage_mt"]

# columns_to_use = [
#     'subducting_plate_absolute_obliquity (degrees)',
#        'subducted_water_volume (m)', 'seafloor_spreading_rate (km/Myr)',
#        'subducting_plate_absolute_velocity_orthogonal (cm/yr)',
#        'subducted_carbonates_volume (m)', 'carbonate_thickness (m)',
#        'trench_velocity_obliquity (degrees)', 'seafloor_age (Ma)',
#        'convergence_obliquity (degrees)', 'trench_velocity_orthogonal (cm/yr)',
#        'co2_volume (m^3/m^2)',
#        'subducting_plate_absolute_velocity_parallel (cm/yr)',
#        'convergence_rate (cm/yr)', 'trench_velocity (cm/yr)',
#        'convergence_rate_parallel (cm/yr)', 'trench_velocity_parallel (cm/yr)',
#        'subducted_sediment_volume (m)',
#        'subducting_plate_absolute_velocity (cm/yr)',
#        'distance_to_trench_edge (degrees)',
#        'convergence_rate_orthogonal (cm/yr)', 'subducted_plate_volume (m)',
#        'sediment_thickness (m)', 'distance_to_trench (km)',
#        'crustal_thickness_mean (m)', 'crustal_thickness_min (m)',
#        'crustal_thickness_max (m)', 'crustal_thickness_median (m)',
#        'crustal_thickness_std (m)', 'crustal_thickness_range (m)',
#        'slab_dip (degrees)', 'arc_trench_distance (km)', 'slab_flux (m^2/yr)',
#        'water_thickness (m)', 'magnetic_anomaly_mean (nT)',
#        'magnetic_anomaly_min (nT)', 'magnetic_anomaly_max (nT)',
#        'magnetic_anomaly_median (nT)', 'magnetic_anomaly_std (nT)',
#        'magnetic_anomaly_range (nT)', 'total_precipitation (km)',
#        'total_convergence (km)', 'fz_distance', 'fz_magnitude',
#        'seamount_distance', 'LIP_distance'
# ]
# ['trench_normal_angle (degrees)', 'distance_from_trench_start (degrees)', 'arc_segment_length (degrees)']

In [ ]:
training_df.columns

In [ ]:
# ==============================================================================
# STEP 2: EXPLORATORY DATA ANALYSIS
# ==============================================================================


training_df=training_df[training_df['present_lon']<=-20]

print("\n" + "="*80)
print("STEP 2: Exploratory Data Analysis")
print("="*80)

# Check for label_binary
if 'label_binary' not in training_df.columns:
    print("\n⚠ WARNING: label_binary column not found!")
    if 'label' in training_df.columns:
        print("  Creating label_binary from label column...")
        training_df['label_binary'] = (training_df['label'] > 0).astype(int)

# PU learning statistics
n_positive = (training_df['label_binary'] == 1).sum()
n_unlabeled = (training_df['label_binary'] == 0).sum()
n_total = len(training_df)

print(f"\nPU Learning Dataset Composition:")
print(f"  Positive samples (known deposits): {n_positive:,}")
print(f"  Unlabeled samples: {n_unlabeled:,}")
print(f"  Total samples: {n_total:,}")
print(f"  Positive rate: {n_positive/n_total*100:.2f}%")

# Validate PU data
diagnostics = validate_pu_data(training_df['label_binary'].values, verbose=True)

# Temporal distribution
print(f"\nTemporal Distribution:")
print(f"  Age range: {training_df['age (Ma)'].min():.1f} - {training_df['age (Ma)'].max():.1f} Ma")
print(f"  Age mean: {training_df['age (Ma)'].mean():.1f} Ma")
print(f"  Age median: {training_df['age (Ma)'].median():.1f} Ma")

# Tonnage statistics
if 'tonnage_mt' in training_df.columns:
    # Handle -9999 or missing values
    tonnage_valid = training_df['tonnage_mt'].replace(-9999, np.nan)
    tonnage_with_values = tonnage_valid[tonnage_valid.notna() & (training_df['label_binary'] == 1)]
    
    print(f"\nTonnage Statistics (known deposits with tonnage):")
    print(f"  Count: {len(tonnage_with_values)}")
    if len(tonnage_with_values) > 0:
        print(f"  Min: {tonnage_with_values.min():.2f} Mt")
        print(f"  Median: {tonnage_with_values.median():.2f} Mt")
        print(f"  Mean: {tonnage_with_values.mean():.2f} Mt")
        print(f"  Max: {tonnage_with_values.max():.2f} Mt")

# Check for missing values
missing_cols = training_df.columns[training_df.isnull().any()].tolist()
if missing_cols:
    print(f"\nColumns with missing values ({len(missing_cols)}):")
    for col in missing_cols[:10]:
        n_missing = training_df[col].isnull().sum()
        print(f"  {col}: {n_missing} ({n_missing/len(training_df)*100:.1f}%)")
    if len(missing_cols) > 10:
        print(f"  ... and {len(missing_cols) - 10} more")

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Label distribution
ax = axes[0, 0]
label_counts = training_df['label_binary'].value_counts()
bars = ax.bar(['Unlabeled', 'Positive'], [label_counts[0], label_counts[1]], 
              color=['blue', 'red'], alpha=0.7, edgecolor='black')
ax.set_ylabel('Count', fontsize=11)
ax.set_title('PU Learning: Dataset Composition', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height):,}\n({height/n_total*100:.1f}%)',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 2: Temporal distribution by class
ax = axes[0, 1]
ax.hist(training_df[training_df['label_binary']==0]['age (Ma)'], bins=30, 
        alpha=0.6, label='Unlabeled', color='blue', density=True)
ax.hist(training_df[training_df['label_binary']==1]['age (Ma)'], bins=30, 
        alpha=0.6, label='Positive (Deposits)', color='red', density=True)
ax.set_xlabel('Age (Ma)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Temporal Distribution by Class', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 3: Spatial distribution
ax = axes[1, 0]
scatter = ax.scatter(training_df[training_df['label_binary']==0]['present_lon'],
                     training_df[training_df['label_binary']==0]['present_lat'],
                     c='blue', alpha=0.3, s=5, label='Unlabeled')
ax.scatter(training_df[training_df['label_binary']==1]['present_lon'],
           training_df[training_df['label_binary']==1]['present_lat'],
           c='red', alpha=0.9, s=50, marker='*', edgecolor='black', linewidth=0.5,
           label='Positive (Deposits)', zorder=5)
ax.set_xlabel('Longitude', fontsize=11)
ax.set_ylabel('Latitude', fontsize=11)
ax.set_title('Spatial Distribution', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 4: Tonnage distribution (if available)
ax = axes[1, 1]
if len(tonnage_with_values) > 0:
    ax.hist(np.log1p(tonnage_with_values), bins=20, 
            color='coral', alpha=0.7, edgecolor='black')
    ax.set_xlabel('Log(1 + Tonnage [Mt])', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title('Deposit Tonnage Distribution', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No tonnage data available', 
            ha='center', va='center', fontsize=12, transform=ax.transAxes)
    ax.axis('off')

plt.tight_layout()
plt.savefig('spatiotemporal_EDA.png', dpi=300, bbox_inches='tight')
print(f"\n✓ EDA plots saved to: spatiotemporal_EDA.png")
plt.show()

In [ ]:
training_df_original=training_df.copy()
training_df=training_df[columns_to_use]


In [ ]:
# ==============================================================================
# STEP 3: FEATURE PREPARATION AND PREPROCESSING
# ==============================================================================
print("\n" + "="*80)
print("STEP 3: Feature Preparation")
print("="*80)

# Define metadata columns (not used as features)
metadata_cols = ['present_lon', 'present_lat', 'lon', 'lat', 'age (Ma)', 
                 'label', 'label_binary', 'tonnage_mt', 'weights', 'distance',
                 'index_right', 'plate_id', 'source', 'region']

# Get feature columns
feature_cols = [col for col in training_df.columns if col not in metadata_cols]

print(f"\nFeature extraction:")
print(f"  Total columns: {len(training_df.columns)}")
print(f"  Metadata columns: {len([c for c in metadata_cols if c in training_df.columns])}")
print(f"  Feature columns: {len(feature_cols)}")


In [ ]:
# ==============================================================================
# STEP 3.5: PHYSICS-AWARE FEATURE GROUPING
# ==============================================================================

import re

feature_groups_patterns = {

    'Plate & Slab Kinematics': [
        r'convergence_rate',
        r'subducting_plate_absolute_velocity',
        r'obliquity'
    ],

    'Trench Kinematics': [
        r'trench_velocity',
        # r'trench_normal_angle',
        r'trench_obliquity',
        r'trench_parallel',
        # r'distance_from_trench_start'
    ],

    'Slab Geometry & Architecture': [
        r'slab_dip',
        r'arc_trench_distance',
        r'arc_segment_length',
        # r'distance_from_trench_start'
    ],

    'Material Input to Trench': [
        r'sediment_thickness',
        r'carbonate_thickness',
        r'water_thickness',
        r'seafloor_age',
         r'^subducted_',
        r'slab_flux',
        r'co2_volume',
        r'seafloor_spreading_rate'
    ],

    'Crustal Thickness': [
        r'crustal_thickness'
    ],

    'Surface Forcing': [
        r'precipitation'
    ],

    'Distance Metrics': [
        r'distance_to_trench (km)',
        # r'fz_distance',
        # r'seamount_distance',
        # r'LIP_distance'
    ],
}

# ------------------------------------------------------------------------------
# Assign features (first match, but clean regex search)
# ------------------------------------------------------------------------------

group_mapping = {}
ungrouped = []

for feature in feature_cols:
    assigned = False
    for group_name, patterns in feature_groups_patterns.items():
        for pattern in patterns:
            if re.search(pattern, feature, re.IGNORECASE):
                group_mapping[feature] = group_name
                assigned = True
                break
        if assigned:
            break

    if not assigned:
        ungrouped.append(feature)

# ------------------------------------------------------------------------------
# Count features
# ------------------------------------------------------------------------------

group_counts = {}
for group in group_mapping.values():
    group_counts[group] = group_counts.get(group, 0) + 1

print("\nFeature distribution across geological process groups:")
for group, count in sorted(group_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {group}: {count}")

if ungrouped:
    print("\nUngrouped features:")
    for f in ungrouped:
        print("  -", f)


# ------------------------------------------------------------------------------
# Visualization
# ------------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(11, 6))
groups = list(group_counts.keys())
counts = list(group_counts.values())
colors = plt.cm.Set3(np.linspace(0, 1, len(groups)))

bars = ax.bar(
    range(len(groups)),
    counts,
    color=colors,
    edgecolor='black',
    linewidth=1,
    alpha=0.85
)

ax.set_xticks(range(len(groups)))
ax.set_xticklabels(groups, rotation=35, ha='right', fontsize=10)
ax.set_ylabel('Number of Features', fontsize=11)
ax.set_title('Feature Distribution by Geodynamics-Based Groups',
             fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add count labels
for bar, count in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        str(count),
        ha='center',
        va='bottom',
        fontsize=9,
        fontweight='bold'
    )

plt.tight_layout()
plt.savefig('geodynamics_feature_groups.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Feature grouping visualization saved to: geodynamics_feature_groups.png")
plt.show()

# ------------------------------------------------------------------------------
# Final note
# ------------------------------------------------------------------------------

print("\n✓ Feature grouping complete")
print("✓ Group labels will be used for:")
print("  - Group-wise importance aggregation")
print("  - SHAP consistency diagnostics")
print("  - Geologically interpretable reporting")


In [ ]:
feature_cols=list(group_mapping.keys())
# Create feature matrix
X_df = training_df[feature_cols].copy()
y = training_df['label_binary'].values

# Handle missing values (simple approach)
print(f"\nHandling missing values...")
n_missing_before = X_df.isnull().sum().sum()
if n_missing_before > 0:
    print(f"  Missing values found: {n_missing_before:,}")
    # Fill with median for numerical columns
    for col in X_df.columns:
        if X_df[col].dtype in ['float64', 'int64']:
            X_df[col].fillna(X_df[col].median(), inplace=True)
        else:
            X_df[col].fillna(0, inplace=True)
    n_missing_after = X_df.isnull().sum().sum()
    print(f"  Missing values after imputation: {n_missing_after}")

# Remove columns with zero variance
print(f"\nRemoving zero-variance features...")
variance = X_df.var()
zero_var_cols = variance[variance == 0].index.tolist()
if len(zero_var_cols) > 0:
    print(f"  Found {len(zero_var_cols)} zero-variance columns")
    X_df = X_df.drop(columns=zero_var_cols)
    print(f"  Features after removal: {X_df.shape[1]}")

# Handle sample weights from tonnage
print(f"\nCreating sample weights from tonnage...")
training_df['tonnage_mt_clean'] = training_df['tonnage_mt'].replace(-9999, np.nan)
positive_mask = training_df['label_binary'] == 1
positive_with_tonnage = positive_mask & training_df['tonnage_mt_clean'].notna()

sample_weights = np.ones(len(training_df))
if positive_with_tonnage.sum() > 0:
    # Use tonnage directly for weighting
    sample_weights[positive_with_tonnage] = training_df.loc[positive_with_tonnage, 'tonnage_mt_clean'].values
    print(f"  Positive samples with tonnage weights: {positive_with_tonnage.sum()}")
    print(f"  Weight range: {sample_weights[positive_with_tonnage].min():.2f} - {sample_weights[positive_with_tonnage].max():.2f}")
else:
    print(f"  No tonnage data - using uniform weights")

print(f"\n✓ Feature preparation complete")
print(f"  Final feature count: {X_df.shape[1]}")
print(f"  Sample count: {X_df.shape[0]:,}")

In [ ]:
# ==============================================================================
# STEP 3.6: REMOVE HIGHLY CORRELATED FEATURES
# ==============================================================================
print("\n" + "="*80)
print("STEP 3.6: Remove Highly Correlated Features")
print("="*80)

print("\nRationale:")
print("  - Highly correlated features provide redundant information")
print("  - Can cause multicollinearity issues in models")
print("  - Reduces computational cost and model complexity")
print("  - Improves model interpretability")

# Configuration
CORRELATION_THRESHOLD = 0.98  # Remove features with correlation > 0.95
# Options: 0.90 (aggressive), 0.95 (balanced), 0.98 (conservative), None (skip)

if CORRELATION_THRESHOLD is not None:
    print(f"\nCorrelation threshold: {CORRELATION_THRESHOLD}")
    print(f"Starting features: {X_df.shape[1]}")
    
    # Compute correlation matrix
    print("\nComputing correlation matrix...")
    corr_matrix = X_df.corr().abs()
    
    # Select upper triangle of correlation matrix (avoid double counting)
    upper_triangle = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )
    
    # Find features with correlation greater than threshold
    to_drop = []
    correlation_pairs = []
    
    for column in upper_triangle.columns:
        # Find correlated features
        correlated_features = upper_triangle.index[upper_triangle[column] > CORRELATION_THRESHOLD].tolist()
        
        if len(correlated_features) > 0:
            for corr_feature in correlated_features:
                corr_value = upper_triangle.loc[corr_feature, column]
                correlation_pairs.append({
                    'feature_1': corr_feature,
                    'feature_2': column,
                    'correlation': corr_value
                })
                
                # Decide which feature to drop (keep the first one encountered)
                if column not in to_drop:
                    to_drop.append(column)
    
    # Create correlation pairs DataFrame
    if len(correlation_pairs) > 0:
        correlation_pairs_df = pd.DataFrame(correlation_pairs)
        correlation_pairs_df = correlation_pairs_df.sort_values('correlation', ascending=False)
        
        print(f"\nFound {len(to_drop)} features to remove (correlation > {CORRELATION_THRESHOLD})")
        print(f"Total correlation pairs found: {len(correlation_pairs)}")
        
        print(f"\nTop 15 highest correlation pairs:")
        print(correlation_pairs_df.head(15).to_string(index=False))
        
        # Save full correlation report
        correlation_pairs_df.to_csv('spatiotemporal_high_correlations.csv', index=False)
        print(f"\n✓ Full correlation report saved to: spatiotemporal_high_correlations.csv")
        
        # Visualize correlation heatmap for highly correlated features
        if len(to_drop) > 0 and len(to_drop) <= 50:  # Only if manageable number
            print(f"\nGenerating correlation heatmap for features to be removed...")
            
            # Get features involved in high correlations
            features_involved = set()
            for _, row in correlation_pairs_df.head(30).iterrows():  # Top 30 pairs
                features_involved.add(row['feature_1'])
                features_involved.add(row['feature_2'])
            
            if len(features_involved) > 1 and len(features_involved) <= 50:
                features_involved = list(features_involved)
                corr_subset = X_df[features_involved].corr()
                
                fig, ax = plt.subplots(figsize=(12, 10))
                sns.heatmap(corr_subset, annot=False, cmap='coolwarm', center=0,
                           square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
                           vmin=-1, vmax=1, ax=ax)
                ax.set_title(f'Correlation Heatmap: Features with Correlation > {CORRELATION_THRESHOLD}',
                           fontsize=13, fontweight='bold', pad=20)
                plt.xticks(rotation=45, ha='right', fontsize=8)
                plt.yticks(rotation=0, fontsize=8)
                plt.tight_layout()
                plt.savefig('spatiotemporal_correlation_heatmap.png', dpi=300, bbox_inches='tight')
                print(f"✓ Correlation heatmap saved to: spatiotemporal_correlation_heatmap.png")
                plt.show()
        
        # Remove highly correlated features
        print(f"\nRemoving {len(to_drop)} highly correlated features...")
        
        # Save dropped features info
        dropped_features_df = pd.DataFrame({
            'dropped_feature': to_drop,
            'reason': 'High correlation (>' + str(CORRELATION_THRESHOLD) + ')'
        })
        dropped_features_df.to_csv('spatiotemporal_dropped_correlated_features.csv', index=False)
        print(f"✓ Dropped features list saved to: spatiotemporal_dropped_correlated_features.csv")
        
        # Drop from X_df
        X_df = X_df.drop(columns=to_drop)
        feature_cols = [col for col in feature_cols if col not in to_drop]
        
        print(f"\n✓ Correlation filtering complete")
        print(f"  Features after removal: {X_df.shape[1]}")
        print(f"  Features removed: {len(to_drop)}")
        print(f"  Reduction: {len(to_drop)/len(feature_cols + to_drop)*100:.1f}%")
        
        # Summary statistics
        print(f"\nCorrelation removal summary:")
        print(f"  Mean correlation (removed pairs): {correlation_pairs_df['correlation'].mean():.3f}")
        print(f"  Max correlation (removed pairs): {correlation_pairs_df['correlation'].max():.3f}")
        print(f"  Min correlation (removed pairs): {correlation_pairs_df['correlation'].min():.3f}")
        
    else:
        print(f"\n✓ No features found with correlation > {CORRELATION_THRESHOLD}")
        print(f"  All features have acceptable correlation levels")
else:
    print(f"\n⚙ Correlation filtering DISABLED (CORRELATION_THRESHOLD = None)")
    print(f"  Keeping all {X_df.shape[1]} features")

print("\n" + "="*80)
print(f"FEATURE PREPROCESSING COMPLETE")
print("="*80)
print(f"  Original features: {len(training_df.columns) - len(metadata_cols)}")
print(f"  After zero-variance removal: {len(feature_cols) + len(to_drop if CORRELATION_THRESHOLD is not None and len(correlation_pairs) > 0 else [])}")
print(f"  After correlation removal: {X_df.shape[1]}")
print(f"  Final feature count: {X_df.shape[1]}")
print("="*80)

---

## 🔍 Correlation Analysis & Feature Redundancy Removal

### Why Remove Highly Correlated Features?

**Multicollinearity Problems:**
- Redundant information (features measure similar things)
- Unstable feature importance estimates
- Increased computational cost
- Reduced model interpretability

**Benefits of Removal:**
- ✅ Faster training time
- ✅ More stable feature importances
- ✅ Better model generalization
- ✅ Easier geological interpretation

### Correlation Threshold Guide

| Threshold | Description | Use Case |
|-----------|-------------|----------|
| **0.90** | Aggressive removal | Maximum feature reduction, computational efficiency |
| **0.95** | Balanced (recommended) | Good trade-off between reduction and information retention |
| **0.98** | Conservative | Keep most features, only remove near-duplicates |
| **None** | Disabled | Skip correlation filtering entirely |

### What Gets Removed?

For each pair of highly correlated features (correlation > threshold):
- The **second feature** encountered is dropped (arbitrary but consistent)
- The **first feature** is kept (preserves ordering)
- All pairwise correlations are saved to CSV for review

### Example Scenarios

**Geophysical data**: Different magnetic/gravity transformations often highly correlated → Safe to remove  
**Derived features**: Distance buffers at different radii often correlated → Consider removing  
**Tectonic variables**: Multiple convergence rate metrics may be redundant → Reduce to most meaningful  

---

**⚙️ Configuration**: Set `CORRELATION_THRESHOLD` in the cell below (default: 0.95)

---

In [ ]:
training_df.columns

In [ ]:
X_df.columns

In [ ]:
# # ==============================================================================
# # STEP 3.5: FEATURE GROUPING (GEOLOGY- & GEODYNAMICS-AWARE ORGANIZATION)
# # ==============================================================================
# print("\n" + "="*80)
# print("STEP 3.5: Feature Grouping (Geology- & Geodynamics-Aware Organization)")
# print("="*80)

# # ------------------------------------------------------------------------------
# # Define geology-aware feature grouping patterns
# # These groups reflect physical processes in subduction systems
# # ------------------------------------------------------------------------------

# # Index(['present_lon', 'present_lat', 'age (Ma)', 'label', 'plate_id', 'lon',
# #        'lat', 'overriding_plate_id', 'source',
# #        'trench_velocity_obliquity (degrees)', 'seafloor_age (Ma)',
# #        'distance_to_trench_edge (degrees)', 'trench_normal_angle (degrees)',
# #        'trench_velocity_orthogonal (cm/yr)', 'subducting_plate_ID',
# #        'subducted_carbonates_volume (m)', 'trench_velocity (cm/yr)',
# #        'subducted_water_volume (m)',
# #        'subducting_plate_absolute_obliquity (degrees)',
# #        'convergence_obliquity (degrees)', 'co2_volume (m^3/m^2)',
# #        'subducted_sediment_volume (m)', 'convergence_rate (cm/yr)',
# #        'sediment_thickness (m)', 'trench_plate_ID', 'carbonate_thickness (m)',
# #        'distance_from_trench_start (degrees)',
# #        'convergence_rate_orthogonal (cm/yr)',
# #        'subducting_plate_absolute_velocity_orthogonal (cm/yr)',
# #        'subducting_plate_absolute_velocity_parallel (cm/yr)',
# #        'subducted_plate_volume (m)', 'convergence_rate_parallel (cm/yr)',
# #        'subducting_plate_absolute_velocity (cm/yr)',
# #        'arc_segment_length (degrees)', 'seafloor_spreading_rate (km/Myr)',
# #        'trench_velocity_parallel (cm/yr)', 'distance_to_trench (km)',
# #        'water_thickness (m)', 'slab_flux (m^2/yr)',
# #        'crustal_thickness_mean (m)', 'crustal_thickness_min (m)',
# #        'crustal_thickness_max (m)', 'crustal_thickness_median (m)',
# #        'crustal_thickness_std (m)', 'crustal_thickness_n',
# #        'crustal_thickness_range (m)', 'magnetic_anomaly_mean (nT)',
# #        'magnetic_anomaly_min (nT)', 'magnetic_anomaly_max (nT)',
# #        'magnetic_anomaly_median (nT)', 'magnetic_anomaly_std (nT)',
# #        'magnetic_anomaly_n', 'magnetic_anomaly_range (nT)',
# #        'slab_dip (degrees)', 'arc_trench_distance (km)',
# #        'total_precipitation (km)', 'total_convergence (km)', 'region',
# #        'fz_distance', 'fz_magnitude', 'seamount_distance', 'LIP_distance',
# #        'tonnage_mt', 'weights', 'label_binary',
# #        'total_precipitation_per_Ma (km/Ma)'],
# #       dtype='object')



# # feature_groups_patterns = {

# #     'Crustal Thickness': [
# #         '*crustal_thickness*'
# #     ],

# #     'Trench Kinematics': [
# #         '*trench_velocity*',
# #         '*trench_normal_angle*',
# #         '*trench_obliquity*',
# #         '*trench_parallel*',
# #         'distance_from_trench_start (degrees)'],
    
# #     'Subducting Plate Kinematics': [
# #         '*subducting_plate*',
# #          'slab_dip (degrees)'  

# #     ],
# #     'Subducted Materials': [
# #         '*subducted*', 'water_thickness (m)', 'slab_flux (m^2/yr)',
# #         'seafloor_age (Ma)', 'co2_volume (m^3/m^2)'
# #         'sediment_thickness (m)', 'carbonate_thickness (m)',
# #          'arc_segment_length (degrees)', 
# #          'seafloor_spreading_rate (km/Myr)'],

# #     'Convergence Dynamics': [
# #         '*convergence*'],
    
# #     'Surface Forcing': [
# #         '*precipitation*'],

# #     'Distance to Trench (km)': [
# #         '*distance_to_trench*']


# # }

# print(f"\nGrouping features into {len(feature_groups_patterns)} geology-aware categories...")

# # ------------------------------------------------------------------------------
# # Assign features to groups using first-match logic
# # ------------------------------------------------------------------------------

# import re
# group_mapping = {}

# for feature in feature_cols:
#     assigned = False
#     for group_name, patterns in feature_groups_patterns.items():
#         for pattern in patterns:
#             regex_pattern = pattern.replace('*', '.*')
#             if re.match(f'^{regex_pattern}$', feature, re.IGNORECASE):
#                 group_mapping[feature] = group_name
#                 assigned = True
#                 break
#         if assigned:
#             break

#     # if not assigned:
#     #     group_mapping[feature] = 'ungrouped'

# # ------------------------------------------------------------------------------
# # Count features per group
# # ------------------------------------------------------------------------------

# group_counts = {}
# for group in group_mapping.values():
#     group_counts[group] = group_counts.get(group, 0) + 1

# print(f"\nFeature distribution across geological groups:")
# for group, count in sorted(group_counts.items(), key=lambda x: x[1], reverse=True):
#     print(f"  {group}: {count} features")

# # ------------------------------------------------------------------------------
# # Visualization
# # ------------------------------------------------------------------------------

# fig, ax = plt.subplots(figsize=(11, 6))
# groups = list(group_counts.keys())
# counts = list(group_counts.values())
# colors = plt.cm.Set3(np.linspace(0, 1, len(groups)))

# bars = ax.bar(
#     range(len(groups)),
#     counts,
#     color=colors,
#     edgecolor='black',
#     linewidth=1,
#     alpha=0.85
# )

# ax.set_xticks(range(len(groups)))
# ax.set_xticklabels(groups, rotation=35, ha='right', fontsize=10)
# ax.set_ylabel('Number of Features', fontsize=11)
# ax.set_title('Feature Distribution by Geodynamics-Based Groups',
#              fontsize=12, fontweight='bold')
# ax.grid(True, alpha=0.3, axis='y')

# # Add count labels
# for bar, count in zip(bars, counts):
#     ax.text(
#         bar.get_x() + bar.get_width() / 2,
#         bar.get_height() + 0.5,
#         str(count),
#         ha='center',
#         va='bottom',
#         fontsize=9,
#         fontweight='bold'
#     )

# plt.tight_layout()
# plt.savefig('geodynamics_feature_groups.png', dpi=300, bbox_inches='tight')
# print(f"\n✓ Feature grouping visualization saved to: geodynamics_feature_groups.png")
# plt.show()

# # ------------------------------------------------------------------------------
# # Final note
# # ------------------------------------------------------------------------------

# print("\n✓ Feature grouping complete")
# print("✓ Group labels will be used for:")
# print("  - Group-wise importance aggregation")
# print("  - SHAP consistency diagnostics")
# print("  - Geologically interpretable reporting")


---

## ⚙️ Configuration Options

### 1. Correlation Filtering (Step 3.6)

Remove highly correlated features to reduce multicollinearity:

```python
# Options:
CORRELATION_THRESHOLD = 0.90   # Aggressive (maximum reduction)
CORRELATION_THRESHOLD = 0.95   # Balanced (recommended)
CORRELATION_THRESHOLD = 0.98   # Conservative (only near-duplicates)
CORRELATION_THRESHOLD = None   # Disabled (keep all features)
```

**When to use:**
- Many derived features → Use 0.95
- Geophysical transformations → Use 0.90-0.95
- Maximum interpretability → Use 0.90
- Keep all information → Use None

---

---

### 2. Feature Grouping (Step 3.5)

Feature grouping is **ENABLED** by default. To customize or disable:

**Option A: Customize grouping patterns**
```python
# Edit Step 3.5 to modify patterns for your dataset
feature_groups_patterns = {
    'your_group': ['*pattern1*', '*pattern2*'],
    # ... add more groups
}
```

**Option B: Skip feature grouping**
```python
# If you skip Step 3.5, features will be marked as 'ungrouped' in importance analysis
# The workflow will still work, but group-level visualizations will show everything as 'ungrouped'
```

**Option C: Simple grouping (no wildcards)**
```python
# Create manual mapping
group_mapping = {
    'feature1': 'group_a',
    'feature2': 'group_a',
    'feature3': 'group_b',
    # ...
}
```

---

### 3. Downsampling (Step 4)

Control class imbalance by downsampling unlabeled samples:

```python
# Options:
DROP_NEGATIVE_FRACTION = None   # No downsampling (default)
DROP_NEGATIVE_FRACTION = 0.4    # Drop 40% of unlabeled
DROP_NEGATIVE_FRACTION = 0.5    # Drop 50% of unlabeled
```

**When to use:**
- Class imbalance > 100:1 → Consider 0.4-0.5
- Computational constraints → Use 0.5
- Maximum information → Use None

---

### 4. 
Optional optimization of XGBoost base estimator:

```python
# Options:
ENABLE_HYPERPARAMETER_TUNING = False  # Use defaults (fast)
ENABLE_HYPERPARAMETER_TUNING = True   # Tune parameters (30-60 min)
```

**When to enable:**
- Final model for publication
- Need maximum performance
- Have computational resources
- Want to understand parameter sensitivity

**What gets tuned:**
- Tree depth, learning rate, regularization
- 50 iterations with 5-fold cross-validation
- Generates sensitivity analysis plots

---

### 5. PU Model Configuration (Step 5)

```python
# Number of bags (more = better uncertainty, slower training)
n_bags = 10  # Fast (default)
n_bags = 50  # Balanced
n_bags = 100 # Slow but best

# Bag size (fraction of training data per bag)
bag_size = 0.7  # Conservative
bag_size = 0.8  # Balanced (default)
bag_size = 0.9  # Aggressive
```

---

In [ ]:
# ==============================================================================
# STEP 4: TRAIN-TEST SPLIT WITH OPTIONAL DOWNSAMPLING
# ==============================================================================
print("\n" + "="*80)
print("STEP 4: Train-Test Split")
print("="*80)

# Stratified split to maintain positive rate
X_train, X_test, y_train, y_test, weights_train, weights_test = train_test_split(
    X_df, y, sample_weights,
    test_size=0.3, 
    stratify=y,
    random_state=RANDOM_STATE
)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"  Positive (deposits): {np.sum(y_train==1):,} ({np.sum(y_train==1)/len(y_train)*100:.2f}%)")
print(f"  Unlabeled: {np.sum(y_train==0):,}")

print(f"\nTest set: {X_test.shape[0]:,} samples")
print(f"  Positive (deposits): {np.sum(y_test==1):,} ({np.sum(y_test==1)/len(y_test)*100:.2f}%)")
print(f"  Unlabeled: {np.sum(y_test==0):,}")

# Optional: Downsample unlabeled samples in training set
DROP_NEGATIVE_FRACTION = None  # Set to 0.4, 0.5, or None

if DROP_NEGATIVE_FRACTION is not None and 0 < DROP_NEGATIVE_FRACTION < 1:
    print(f"\n⚙ Downsampling training set: Dropping {DROP_NEGATIVE_FRACTION*100:.0f}% of unlabeled")
    
    positive_train_idx = np.where(y_train == 1)[0]
    negative_train_idx = np.where(y_train == 0)[0]
    n_negative_keep = int(len(negative_train_idx) * (1 - DROP_NEGATIVE_FRACTION))
    
    np.random.seed(RANDOM_STATE)
    negative_keep_idx = np.random.choice(negative_train_idx, size=n_negative_keep, replace=False)
    keep_train_idx = np.concatenate([positive_train_idx, negative_keep_idx])
    np.random.shuffle(keep_train_idx)
    
    X_train = X_train.iloc[keep_train_idx].reset_index(drop=True)
    y_train = y_train[keep_train_idx]
    weights_train = weights_train[keep_train_idx]
    
    print(f"  After downsampling: {len(y_train):,} samples")
    print(f"  Positive: {np.sum(y_train==1):,} ({np.sum(y_train==1)/len(y_train)*100:.2f}%)")
else:
    print(f"\n⚙ No downsampling (keeping all training samples)")

print("\n" + "="*80)

---

## 🎛️ Hyperparameter Tuning (Optional)

### Should You Tune?

**Skip tuning if:**
- ⏭️ First iteration / exploratory analysis
- ⏭️ Default parameters work reasonably well
- ⏭️ Time constraints (tuning can take 30-60 minutes)

**Use tuning if:**
- ✅ Optimizing final model for publication
- ✅ Need maximum performance
- ✅ Want to understand hyperparameter sensitivity
- ✅ Have computational resources available

### What Gets Tuned?

**XGBoost Base Estimator Parameters:**
- `max_depth`: Tree depth (complexity control)
- `learning_rate`: Step size shrinkage
- `min_child_weight`: Minimum sum of instance weight
- `subsample`: Fraction of samples per tree
- `colsample_bytree`: Fraction of features per tree
- `reg_alpha`: L1 regularization
- `reg_lambda`: L2 regularization

**Strategy**: 5-fold stratified cross-validation on training set with AUC-ROC scoring

---

**⚙️ Set `ENABLE_HYPERPARAMETER_TUNING = True` in the next cell to enable**

---

In [ ]:
# ==============================================================================
# STEP 4.5: HYPERPARAMETER TUNING (OPTIONAL)
# ==============================================================================
print("\n" + "="*80)
print("STEP 4.5: Hyperparameter Tuning (Optional)")
print("="*80)

# Configuration
ENABLE_HYPERPARAMETER_TUNING = True  # Set to True to enable tuning
# Options: True (tune hyperparameters), False (use defaults)

if ENABLE_HYPERPARAMETER_TUNING:
    from sklearn.model_selection import RandomizedSearchCV
    from scipy.stats import uniform, randint
    
    print("\n⚙ Hyperparameter tuning ENABLED")
    print("  This will take 30-60 minutes depending on dataset size...")
    print("  Using Randomized Search with 5-fold stratified cross-validation")
    
    # Define parameter distributions for randomized search
    # param_distributions = {
    #     'max_depth': randint(3, 10),           # Tree depth
    #     'learning_rate': uniform(0.01, 0.19),  # 0.01 to 0.20
    #     'min_child_weight': randint(1, 7),     # Minimum sum of instance weight
    #     'subsample': uniform(0.7, 0.3),        # 0.7 to 1.0
    #     'colsample_bytree': uniform(0.7, 0.3), # 0.7 to 1.0
    #     'reg_alpha': uniform(0, 0.5),          # L1 regularization
    #     'reg_lambda': uniform(0.1, 1.9),       # L2 regularization 0.1 to 2.0
    # }

#     param_distributions = {
#     # Increase the upper bound for depth
#     'max_depth': randint(5, 12),           
    
#     # Focus on the high-performing learning rate zone
#     'learning_rate': uniform(0.05, 0.2),  
    
#     # Lower weight to allow more complex trees (based on your red bar)
#     'min_child_weight': randint(1, 4),     
    
#     'subsample': uniform(0.6, 0.4),        
#     'colsample_bytree': uniform(0.6, 0.4), 
#     'reg_alpha': uniform(0, 0.8),          
#     'reg_lambda': uniform(0.5, 2.5),       
# }
    # Calculate this ratio beforehand
    ratio = float(sum(y == 0)) / sum(y == 1)
    
    param_distributions = {
    # 'scale_pos_weight': uniform(ratio * 0.8, ratio * 1.2),
    # # Narrowing the learning rate to the 'sweet spot'
    # 'learning_rate': uniform(0.03, 0.05),  
    
    # # Keeping depth moderate but pushing max_depth slightly if needed
    # 'max_depth': randint(4, 7),           
    
    # # Stay firm on min_child_weight to protect against imbalance noise
    # 'min_child_weight': randint(8, 12),     
    
    # # High sampling since it showed positive correlation
    # 'subsample': uniform(0.8, 0.15),        
    # 'colsample_bytree': uniform(0.8, 0.15), 
    
    # # Let's try to increase L1 (alpha) slightly to prune useless features
    # 'reg_alpha': uniform(0.5, 2.0),          
    # 'reg_lambda': uniform(10, 20), # Keep L2 high for stability
    # Narrowing depth to prevent the 1.0 train score
    'max_depth': randint(4, 9),           
    
    # Lower learning rate usually requires more n_estimators
    'learning_rate': uniform(0.01, 0.1),  
    
    # Keeping this low as per your sensitivity plot
    'min_child_weight': randint(1, 3),     
    
    # Higher subsample and colsample based on positive correlation
    'subsample': uniform(0.8, 0.2),        
    'colsample_bytree': uniform(0.8, 0.2), 
    
    # Significantly bumping regularization to fix the overfitting plot
    'reg_alpha': uniform(0, 2.0),          
    'reg_lambda': uniform(1.0, 10.0),       
}
    # Base estimator for tuning
    base_estimator_tuning = xgb.XGBClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=1,
        eval_metric='logloss',
        use_label_encoder=False
    )
    
    print(f"\nParameter search space:")
    for param, dist in param_distributions.items():
        if hasattr(dist, 'low') and hasattr(dist, 'high'):
            print(f"  {param}: [{dist.low:.2f}, {dist.high:.2f}]")
        elif hasattr(dist, 'a') and hasattr(dist, 'b'):
            print(f"  {param}: [{dist.a}, {dist.b}]")
    
    # Randomized search
    print(f"\nStarting randomized search...")
    print(f"  Number of iterations: 50")
    print(f"  Cross-validation folds: 5")
    print(f"  Scoring metric: AUC-ROC")
    
    random_search = RandomizedSearchCV(
        estimator=base_estimator_tuning,
        param_distributions=param_distributions,
        n_iter=50,  # Number of parameter settings sampled
        scoring='roc_auc',
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=2,
        return_train_score=True
    )
    
    from sklearn.impute import SimpleImputer

    # imputer = SimpleImputer(strategy='median')
    # X_train = imputer.fit_transform(X_train)
    # Fit with sample weights
    random_search.fit(X_train, y_train, sample_weight=weights_train)
    
    # Get best parameters
    best_params = random_search.best_params_
    best_score = random_search.best_score_
    
    print(f"\n{'='*80}")
    print("HYPERPARAMETER TUNING RESULTS")
    print(f"{'='*80}")
    print(f"\nBest cross-validation AUC-ROC: {best_score:.4f}")
    print(f"\nBest hyperparameters:")
    for param, value in best_params.items():
        print(f"  {param}: {value}")
    
    # Analyze parameter importance via variance in scores
    results_df = pd.DataFrame(random_search.cv_results_)
    
    print(f"\n{'='*80}")
    print("HYPERPARAMETER SENSITIVITY ANALYSIS")
    print(f"{'='*80}")
    
    # Correlation of parameters with performance
    param_cols = [col for col in results_df.columns if col.startswith('param_')]
    score_col = 'mean_test_score'
    
    print(f"\nParameter impact on performance (correlation with AUC-ROC):")
    correlations = {}
    for col in param_cols:
        param_name = col.replace('param_', '')
        # Convert to numeric if possible
        try:
            param_values = pd.to_numeric(results_df[col])
            corr = param_values.corr(results_df[score_col])
            correlations[param_name] = corr
            print(f"  {param_name}: {corr:+.3f} {'(+increase helps)' if corr > 0.1 else '(-decrease helps)' if corr < -0.1 else '(minimal impact)'}")
        except:
            pass
    
    # Show top 10 parameter combinations
    print(f"\nTop 10 parameter combinations:")
    top_results = results_df.nsmallest(10, 'rank_test_score')[
        ['rank_test_score', 'mean_test_score', 'std_test_score'] + param_cols
    ]
    print(top_results.to_string(index=False))
    
    # Visualize tuning results
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: Score distribution
    ax = axes[0, 0]
    ax.hist(results_df['mean_test_score'], bins=30, edgecolor='black', alpha=0.7, color='skyblue')
    ax.axvline(best_score, color='red', linestyle='--', linewidth=2, label=f'Best: {best_score:.4f}')
    ax.set_xlabel('Cross-Validation AUC-ROC', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title('Distribution of Cross-Validation Scores', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Score vs iterations
    ax = axes[0, 1]
    iterations = range(len(results_df))
    ax.plot(iterations, results_df['mean_test_score'], 'o-', alpha=0.6, markersize=4)
    ax.axhline(best_score, color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax.set_xlabel('Iteration', fontsize=11)
    ax.set_ylabel('Cross-Validation AUC-ROC', fontsize=11)
    ax.set_title('Hyperparameter Search Progress', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Plot 3: Parameter importance (bar chart of correlations)
    ax = axes[1, 0]
    if correlations:
        sorted_params = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)
        param_names = [p[0] for p in sorted_params]
        param_corrs = [p[1] for p in sorted_params]
        colors = ['green' if c > 0 else 'red' for c in param_corrs]
        
        y_pos = np.arange(len(param_names))
        ax.barh(y_pos, param_corrs, color=colors, alpha=0.7, edgecolor='black')
        ax.set_yticks(y_pos)
        ax.set_yticklabels(param_names, fontsize=10)
        ax.set_xlabel('Correlation with AUC-ROC', fontsize=11)
        ax.set_title('Parameter Sensitivity', fontsize=12, fontweight='bold')
        ax.axvline(0, color='black', linewidth=1)
        ax.grid(True, alpha=0.3, axis='x')
    else:
        ax.text(0.5, 0.5, 'Correlation analysis unavailable', 
               ha='center', va='center', transform=ax.transAxes, fontsize=12)
        ax.axis('off')
    
    # Plot 4: Train vs Test score
    ax = axes[1, 1]
    ax.scatter(results_df['mean_train_score'], results_df['mean_test_score'], 
              alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
    
    # Diagonal line (perfect generalization)
    min_score = min(results_df['mean_train_score'].min(), results_df['mean_test_score'].min())
    max_score = max(results_df['mean_train_score'].max(), results_df['mean_test_score'].max())
    ax.plot([min_score, max_score], [min_score, max_score], 'r--', linewidth=2, alpha=0.7, label='Perfect Generalization')
    
    ax.set_xlabel('Mean Train Score (AUC-ROC)', fontsize=11)
    ax.set_ylabel('Mean Test Score (AUC-ROC)', fontsize=11)
    ax.set_title('Overfitting Analysis', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('spatiotemporal_hyperparameter_tuning.png', dpi=300, bbox_inches='tight')
    print(f"\n✓ Tuning analysis plots saved to: spatiotemporal_hyperparameter_tuning.png")
    plt.show()
    
    # Save tuning results
    results_df.to_csv('spatiotemporal_hyperparameter_tuning_results.csv', index=False)
    print(f"✓ Full tuning results saved to: spatiotemporal_hyperparameter_tuning_results.csv")
    
    # Save best parameters
    best_params_df = pd.DataFrame([best_params])
    best_params_df['best_cv_score'] = best_score
    best_params_df.to_csv('spatiotemporal_best_hyperparameters.csv', index=False)
    print(f"✓ Best parameters saved to: spatiotemporal_best_hyperparameters.csv")
    
    print("\n" + "="*80)
    print("RECOMMENDATION")
    print("="*80)
    print("Use the best hyperparameters in Step 5 (Model Training)")
    print("Replace default values with tuned values shown above")
    print("="*80)
    
else:
    print("\n⚙ Hyperparameter tuning DISABLED")
    print("  Using default XGBoost parameters")
    print("  Set ENABLE_HYPERPARAMETER_TUNING = True to enable tuning")
    
    # Default parameters
    best_params = {
        'max_depth': 6,
        'learning_rate': 0.1,
        'min_child_weight': 3,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0
    }
    
    print(f"\nUsing default parameters:")
    for param, value in best_params.items():
        print(f"  {param}: {value}")

print("\n" + "="*80)

In [ ]:
# ==============================================================================
# STEP 5: TRAIN PU BAGGING CLASSIFIER
# ==============================================================================
print("\n" + "="*80)
print("STEP 5: Train PU Bagging Classifier")
print("="*80)

# Configure base estimator (XGBoost) - uses best_params from Step 4.5
print(f"Using hyperparameters from Step 4.5 (tuned if enabled, defaults otherwise)")

base_estimator = xgb.XGBClassifier(
    max_depth=best_params['max_depth'],
    learning_rate=best_params['learning_rate'],
    n_estimators=100,
    min_child_weight=best_params['min_child_weight'],
    subsample=best_params['subsample'],
    colsample_bytree=best_params['colsample_bytree'],
    reg_alpha=best_params['reg_alpha'],
    reg_lambda=best_params['reg_lambda'],
    random_state=RANDOM_STATE,
    n_jobs=-1,
    eval_metric='logloss',
    use_label_encoder=False
)

print(f"\nBase estimator: XGBoost")
print(f"  max_depth: {base_estimator.max_depth}")
print(f"  learning_rate: {base_estimator.learning_rate}")
print(f"  n_estimators: {base_estimator.n_estimators}")
print(f"  min_child_weight: {base_estimator.min_child_weight}")
print(f"  subsample: {base_estimator.subsample}")
print(f"  colsample_bytree: {base_estimator.colsample_bytree}")
print(f"  reg_alpha: {base_estimator.reg_alpha}")
print(f"  reg_lambda: {base_estimator.reg_lambda}")

# Initialize PU Bagging Classifier
print(f"\nInitializing PU Bagging Classifier...")
pu_model = PUBaggingClassifier(
    base_estimator=base_estimator,
    n_bags=100,  # Number of bags
    bag_size=0.8,  # 80% of training data per bag
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

print(f"  Number of bags: {pu_model.n_bags}")
print(f"  Samples per bag: {pu_model.bag_size}")

# Train the model
print(f"\nTraining PU Bagging Classifier...")
print(f"  This may take several minutes...")

pu_model.fit(X_train, y_train, sample_weight=weights_train)

print(f"\n✓ Model training complete")
print(f"  Total bags trained: {len(pu_model.estimators_)}")
print(f"  Training samples: {len(y_train):,}")
print(f"  Features used: {X_train.shape[1]}")

In [ ]:
# ==============================================================================
# STEP 6: PREDICTION AND PU-AWARE EVALUATION
# ==============================================================================
print("\n" + "="*80)
print("STEP 6: Prediction with Uncertainty Quantification")
print("="*80)

# Predict with uncertainty
y_pred_mean, y_pred_std = pu_model.predict_with_uncertainty(X_test)

print(f"Prediction statistics:")
print(f"  Mean probability: {y_pred_mean.mean():.3f}")
print(f"  Std probability: {y_pred_mean.std():.3f}")
print(f"  Mean uncertainty: {y_pred_std.mean():.3f}")
print(f"  Max uncertainty: {y_pred_std.max():.3f}")

# Standard metrics (reference only - treats unlabeled as negative)
auc_roc = roc_auc_score(y_test, y_pred_mean)
auc_pr = average_precision_score(y_test, y_pred_mean)

print(f"\n⚠ Standard metrics (reference only - treats unlabeled as negative):")
print(f"  AUC-ROC: {auc_roc:.3f}")
print(f"  AUC-PR: {auc_pr:.3f}")

print(f"\nPrediction distribution by class:")
print(f"  Known deposits - Mean: {y_pred_mean[y_test==1].mean():.3f}, Std: {y_pred_mean[y_test==1].std():.3f}")
print(f"  Unlabeled - Mean: {y_pred_mean[y_test==0].mean():.3f}, Std: {y_pred_mean[y_test==0].std():.3f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Prediction distribution
ax = axes[0]
ax.hist(y_pred_mean[y_test==0], bins=50, alpha=0.6, label='Unlabeled', color='blue', density=True)
ax.hist(y_pred_mean[y_test==1], bins=50, alpha=0.6, label='Known Deposits', color='red', density=True)
ax.set_xlabel('Predicted Probability', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Prediction Distribution', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Uncertainty vs prediction
ax = axes[1]
scatter = ax.scatter(y_pred_mean, y_pred_std, c=y_test, cmap='coolwarm', 
                    alpha=0.5, edgecolors='black', linewidth=0.3, s=20)
ax.set_xlabel('Mean Prediction', fontsize=11)
ax.set_ylabel('Prediction Uncertainty (Std)', fontsize=11)
ax.set_title('Uncertainty Analysis', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_ticks([0, 1])
cbar.set_ticklabels(['Unlabeled', 'Positive'])

plt.tight_layout()
plt.savefig('spatiotemporal_predictions.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Prediction plots saved to: spatiotemporal_predictions.png")
plt.show()

In [ ]:
# ==============================================================================
# STEP 7: PU-AWARE EVALUATION METRICS
# ==============================================================================
print("\n" + "="*80)
print("STEP 7: PU-Aware Evaluation Metrics")
print("="*80)

print("\n⚠ PU Learning Evaluation Strategy:")
print("  - Positive samples (y=1) = KNOWN deposits")
print("  - Unlabeled samples (y=0) = UNKNOWN (may contain hidden deposits)")
print("  - We evaluate ONLY on known deposits' rankings")

# Get deposit statistics
n_total = len(y_pred_mean)
n_deposits = np.sum(y_test == 1)
deposit_mask = y_test == 1
deposit_predictions = y_pred_mean[deposit_mask]

print(f"\nTest set statistics:")
print(f"  Total samples: {n_total:,}")
print(f"  Known deposits: {n_deposits}")
print(f"  Unlabeled: {n_total - n_deposits:,}")

# ==============================================================================
# 1. RANK ANALYSIS - WHERE DO KNOWN DEPOSITS APPEAR IN RANKING?
# ==============================================================================
print("\n" + "-"*80)
print("Ranking Analysis: Position of Known Deposits")
print("-"*80)

# Rank all samples by prediction score (highest to lowest)
all_ranked_indices = np.argsort(y_pred_mean)[::-1]

# Find ranks of each known deposit
deposit_indices = np.where(deposit_mask)[0]
deposit_ranks = []
for dep_idx in deposit_indices:
    rank = np.where(all_ranked_indices == dep_idx)[0][0] + 1
    percentile = (rank / n_total) * 100
    deposit_ranks.append({
        'deposit_idx': dep_idx,
        'rank': rank,
        'percentile': percentile,
        'prediction_score': y_pred_mean[dep_idx]
    })

deposit_ranks_df = pd.DataFrame(deposit_ranks)
deposit_ranks_df = deposit_ranks_df.sort_values('rank')

print(f"\nKnown deposit rankings:")
print(f"  Best ranked deposit: #{deposit_ranks_df['rank'].min()} (top {deposit_ranks_df['percentile'].min():.2f}%)")
print(f"  Median ranked deposit: #{deposit_ranks_df['rank'].median():.0f} (top {deposit_ranks_df['percentile'].median():.1f}%)")
print(f"  Worst ranked deposit: #{deposit_ranks_df['rank'].max():.0f} (top {deposit_ranks_df['percentile'].max():.1f}%)")

print(f"\nTop 10 highest-ranked deposits:")
print(deposit_ranks_df.head(10)[['rank', 'percentile', 'prediction_score']].to_string(index=False))

# ==============================================================================
# 2. RECALL@K AND ENRICHMENT METRICS
# ==============================================================================
print("\n" + "-"*80)
print("Computing Recall@K and Enrichment Metrics")
print("-"*80)

percentiles = [1, 2, 5, 10, 20, 30, 50]
metrics_at_k = []

print(f"\nRecall@K: Deposits Captured in Top-K% Ranked Areas")
print("-"*80)

for pct in percentiles:
    k = int(n_total * pct / 100)
    if k == 0:
        k = 1
    
    top_k_indices = all_ranked_indices[:k]
    deposits_in_top_k = np.sum(deposit_mask[top_k_indices])
    
    # Recall@K: proportion of all known deposits captured in top-k
    recall_k = deposits_in_top_k / n_deposits if n_deposits > 0 else 0
    
    # Enrichment factor: how many times more deposits than random
    expected_random = n_deposits * (pct / 100)
    enrichment = deposits_in_top_k / expected_random if expected_random > 0 else 0
    
    metrics_at_k.append({
        'Area %': pct,
        'K (samples)': k,
        'Deposits Captured': int(deposits_in_top_k),
        'Recall@K': recall_k,
        'Enrichment': enrichment
    })
    
    print(f"Top {pct:2}% (K={k:,}): {int(deposits_in_top_k):2}/{n_deposits} deposits ({recall_k*100:5.1f}%) - Enrichment: {enrichment:5.2f}x")

metrics_df = pd.DataFrame(metrics_at_k)

# Key finding
top5_metrics = metrics_df[metrics_df['Area %'] == 5].iloc[0]
print(f"\n" + "="*80)
print(f"KEY FINDING (PU-AWARE)")
print("="*80)
print(f"The top 5% of ranked area captures {int(top5_metrics['Deposits Captured'])} out of {n_deposits} ")
print(f"known deposits ({top5_metrics['Recall@K']*100:.1f}%), with an enrichment of {top5_metrics['Enrichment']:.1f}x")
print("="*80)

# ==============================================================================
# 3. SUCCESS-RATE CURVES (CUMULATIVE DEPOSITS VS AREA)
# ==============================================================================
print("\n" + "-"*80)
print("Computing Success-Rate Curves (Based on Known Deposits)")
print("-"*80)

# Compute cumulative known deposits as we go down the ranked list
y_test_array = y_test.values if hasattr(y_test, 'values') else y_test
cumulative_deposits = np.cumsum(y_test_array[all_ranked_indices])
cumulative_area_pct = np.arange(1, n_total + 1) / n_total * 100

# Random baseline
random_baseline = np.linspace(0, n_deposits, n_total)

# Compute success rate at various area percentages
success_rate_points = []
for area_pct in [1, 2, 5, 10, 20, 30, 50, 75, 100]:
    idx = int(n_total * area_pct / 100) - 1
    if idx >= len(cumulative_deposits):
        idx = len(cumulative_deposits) - 1
    
    deposits_at_pct = cumulative_deposits[idx]
    success_rate = (deposits_at_pct / n_deposits * 100) if n_deposits > 0 else 0
    
    success_rate_points.append({
        'Area %': area_pct,
        'Deposits Found': int(deposits_at_pct),
        'Success Rate %': success_rate
    })

success_rate_df = pd.DataFrame(success_rate_points)
print("\nSuccess Rate at Key Area Percentiles:")
print(success_rate_df.to_string(index=False))

# Save metrics
metrics_df.to_csv('spatiotemporal_PU_metrics.csv', index=False)
success_rate_df.to_csv('spatiotemporal_success_rate_curve_data.csv', index=False)
deposit_ranks_df.to_csv('spatiotemporal_deposit_rankings.csv', index=False)
print(f"\n✓ Metrics saved to: spatiotemporal_PU_metrics.csv")
print(f"✓ Success-rate data saved to: spatiotemporal_success_rate_curve_data.csv")
print(f"✓ Deposit rankings saved to: spatiotemporal_deposit_rankings.csv")

In [ ]:
# ==============================================================================
# STEP 7.5: COMPREHENSIVE VISUALIZATION - SUCCESS-RATE CURVES
# ==============================================================================
print("\n" + "="*80)
print("STEP 7.5: Comprehensive Visualization - Success-Rate Curves")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ==============================================================================
# Plot 1: Success-Rate Curve (Cumulative Deposits vs Area)
# ==============================================================================
ax = axes[0, 0]
ax.plot(cumulative_area_pct, cumulative_deposits, 'b-', linewidth=2.5, label='PU Model', zorder=3)
ax.plot(cumulative_area_pct, random_baseline, 'r--', linewidth=1.5, label='Random Baseline', alpha=0.7)
ax.fill_between(cumulative_area_pct, random_baseline, cumulative_deposits, 
                 where=(cumulative_deposits >= random_baseline), 
                 alpha=0.3, color='green', label='Model Gain')
ax.set_xlabel('Cumulative Area Explored (%)', fontsize=12)
ax.set_ylabel('Cumulative Deposits Found', fontsize=12)
ax.set_title('Success-Rate Curve', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 100])
ax.set_ylim([0, n_deposits * 1.05])

# Add annotation for top 5%
top5_idx = int(n_total * 0.05)
top5_deposits = cumulative_deposits[top5_idx - 1]
ax.plot([5, 5], [0, top5_deposits], 'g--', linewidth=1.5, alpha=0.7)
ax.plot([0, 5], [top5_deposits, top5_deposits], 'g--', linewidth=1.5, alpha=0.7)
ax.text(5, top5_deposits + n_deposits*0.02, 
        f'{int(top5_deposits)} deposits\nat 5% area', 
        fontsize=9, color='green', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='green', alpha=0.8))

# ==============================================================================
# Plot 2: Lift Curve
# ==============================================================================
ax = axes[0, 1]
lift_curve = (cumulative_deposits / np.arange(1, n_total + 1)) / (n_deposits / n_total)
# Only plot up to 50% area for clarity
plot_limit = int(n_total * 0.5)
ax.plot(cumulative_area_pct[:plot_limit], lift_curve[:plot_limit], 'purple', linewidth=2.5)
ax.axhline(y=1, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Random (Lift=1)')
ax.set_xlabel('Cumulative Area Explored (%)', fontsize=12)
ax.set_ylabel('Lift over Random', fontsize=12)
ax.set_title('Lift Curve', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 50])

# Add annotations for key percentiles
for pct in [1, 2, 5, 10]:
    idx = int(n_total * pct / 100) - 1
    lift_val = lift_curve[idx]
    ax.plot(pct, lift_val, 'ro', markersize=8, zorder=5)
    ax.text(pct, lift_val + 0.5, f'{lift_val:.1f}x\n@{pct}%', 
            fontsize=8, ha='center', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.6))

# ==============================================================================
# Plot 3: Recall@K and Enrichment Curves (PU-Aware)
# ==============================================================================
ax = axes[1, 0]
area_range = np.linspace(0, 30, 100)
recall_curve = []
enrichment_curve = []

for pct in area_range:
    k = int(n_total * pct / 100)
    if k == 0 or pct == 0:
        recall_curve.append(0)
        enrichment_curve.append(0)
    else:
        top_k_indices = all_ranked_indices[:k]
        deposits_in_top_k = np.sum(y_test_array[top_k_indices])
        recall_curve.append(deposits_in_top_k / n_deposits if n_deposits > 0 else 0)
        expected_random = n_deposits * (pct / 100)
        enrichment_curve.append(deposits_in_top_k / expected_random if expected_random > 0 else 0)

ax.plot(area_range, recall_curve, 'r-', linewidth=2.5, label='Recall@K', marker='s', 
        markevery=10, markersize=5)
ax2 = ax.twinx()
ax2.plot(area_range, enrichment_curve, 'purple', linewidth=2.5, label='Enrichment Factor', marker='o', 
         markevery=10, markersize=5, alpha=0.7)
ax2.axhline(y=1, color='gray', linestyle='--', linewidth=1.5, alpha=0.5, label='Random (Enrichment=1)')
ax.set_xlabel('Top Area (%) Selected', fontsize=12)
ax.set_ylabel('Recall@K', fontsize=12, color='red')
ax2.set_ylabel('Enrichment Factor', fontsize=12, color='purple')
ax.set_title('Recall@K and Enrichment', fontsize=13, fontweight='bold')
ax.tick_params(axis='y', labelcolor='red')
ax2.tick_params(axis='y', labelcolor='purple')
ax.legend(fontsize=10, loc='upper left')
ax2.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 30])
ax.set_ylim([0, 1.05])

# Add vertical lines for key percentiles
for pct in [1, 2, 5, 10]:
    ax.axvline(x=pct, color='green', linestyle=':', alpha=0.5, linewidth=1)

# ==============================================================================
# Plot 4: Summary Bar Chart - Deposits Captured at Key Percentiles
# ==============================================================================
ax = axes[1, 1]
bar_data = metrics_df[metrics_df['Area %'] <= 20]  # Focus on top 20%
x_pos = np.arange(len(bar_data))
bars = ax.bar(x_pos, bar_data['Deposits Captured'], alpha=0.8, 
              edgecolor='black', linewidth=1.5, color='steelblue')
ax.set_xticks(x_pos)
ax.set_xticklabels([f"{int(pct)}%" for pct in bar_data['Area %']], fontsize=11)
ax.set_xlabel('Top Area Percentile', fontsize=12)
ax.set_ylabel('Number of Deposits Captured', fontsize=12)
ax.set_title(f'Deposits Captured at Key Percentiles\n(Out of {n_deposits} Total)', 
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for i, bar in enumerate(bars):
    height = bar.get_height()
    recall_val = bar_data.iloc[i]['Recall@K']
    enrichment_val = bar_data.iloc[i]['Enrichment']
    
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.3,
            f'{int(height)}\n({recall_val*100:.1f}%)',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # Add enrichment annotation inside bar
    if height > 0:
        ax.text(bar.get_x() + bar.get_width()/2., height/2,
                f'{enrichment_val:.1f}x',
                ha='center', va='center', fontsize=8, color='white', fontweight='bold')

ax.set_ylim([0, n_deposits * 1.15])

plt.tight_layout()
plt.savefig('spatiotemporal_prospectivity_evaluation.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Success-rate curves saved to: spatiotemporal_prospectivity_evaluation.png")
plt.show()

print("\n" + "="*80)
print("PU-Aware Mineral Prospectivity Evaluation Complete")
print("="*80)
print("\nKey Points:")
print("  ✓ Evaluation focuses on ranking known deposits highly")
print("  ✓ Unlabeled samples are NOT treated as negatives")
print("  ✓ Enrichment factor shows how much better than random")
print("  ✓ High-scoring unlabeled areas may contain undiscovered deposits")

In [ ]:
# ==============================================================================
# STEP 8: FEATURE IMPORTANCE ANALYSIS WITH GROUPING
# ==============================================================================
print("\n" + "="*80)
print("STEP 8: Feature Importance Analysis")
print("="*80)

# Get feature importances
importance_mean, importance_std = pu_model.get_feature_importances()


# Create importance DataFrame
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance_mean': importance_mean,
    'importance_std': importance_std,
    'cv': importance_std / (importance_mean + 1e-10)
})

# --- NEW: Create a clean display name for plots and tables ---
importance_df['display_name'] = (
    importance_df['feature']
    .str.replace('_', ' ')           # Remove underscores
    .str.title()                     # Capitalize each word
    .str.replace('Km', 'km')         # Optional: keep units lowercase
    .str.replace('Ma', 'Ma')
)

# Assign geological groups
importance_df['group'] = importance_df['feature'].map(group_mapping)
importance_df = importance_df.sort_values('importance_mean', ascending=False)

# --- IMPROVED: Create display names with lowercase units ---
importance_df['display_name'] = (
    importance_df['feature']
    .str.replace('_', ' ')           # Remove underscores
    .str.title()                     # Initial capitalization
)

# Fix specific geological/scientific units to be lowercase/standard
unit_fixes = {
    ' (M)': ' (m)',
    ' (Km)': ' (km)',
    ' (Cm/Yr)': ' (cm/yr)',
    ' (M^2/Yr)': ' (m²/yr)',
    ' (Degrees)': ' (degrees)',
    ' (Ma)': ' (Ma)',                # Mega-annum (Standard notation)
    ' (Myr)': ' (Myr)',              # Million years (Standard notation)
    ' (Kj/M^2)': ' (kJ/m²)'          # For any energy/forcing features
}

for search, replace in unit_fixes.items():
    importance_df['display_name'] = importance_df['display_name'].str.replace(search, replace, regex=False)



print(f"\nTop 20 most important features:")
# Use 'display_name' in the print statement
print(importance_df.head(20)[['display_name', 'importance_mean', 'importance_std', 'group']].to_string(index=False))



# ... (Group-level importance code remains the same) ...

# ==============================================================================
# VISUALIZATION: Feature Importance
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 7),dpi=300)

# Plot 1: Top individual features
ax = axes[0]
top_n = 5
top_features = importance_df.head(top_n)
y_pos = np.arange(len(top_features))
colors = plt.cm.viridis(np.linspace(0, 1, top_n))

ax.barh(y_pos, top_features['importance_mean'], 
        xerr=top_features['importance_std'],
        alpha=0.8, edgecolor='black', color=colors, linewidth=0.5)
ax.set_yticks(y_pos)



# CHANGE: Use 'display_name' for the tick labels
ax.set_yticklabels(top_features['display_name'], fontsize=8) 

ax.set_xlabel('Mean Importance', fontsize=11)
ax.set_title(f'Top {top_n} Most Important Features', fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')


# # Create importance DataFrame
# importance_df = pd.DataFrame({
#     'feature': X_train.columns,
#     'importance_mean': importance_mean,
#     'importance_std': importance_std,
#     'cv': importance_std / (importance_mean + 1e-10)
# })

# # Assign geological groups (using group_mapping from Step 3.5)
# importance_df['group'] = importance_df['feature'].map(group_mapping)

# importance_df = importance_df.sort_values('importance_mean', ascending=False)

# print(f"\nTop 20 most important features:")
# print(importance_df.head(20)[['feature', 'importance_mean', 'importance_std', 'group']].to_string(index=False))

# # Group-level importance
print(f"\n" + "-"*80)
print("Feature Importance by Geological Group")
print("-"*80)
group_importance = importance_df.groupby('group')['importance_mean'].sum().sort_values(ascending=False)
print(group_importance.to_string())

# Save full report
importance_df.to_csv('spatiotemporal_feature_importance.csv', index=False)
group_importance.to_csv('spatiotemporal_group_importance.csv')
print(f"\n✓ Feature importance saved to: spatiotemporal_feature_importance.csv")
print(f"✓ Group importance saved to: spatiotemporal_group_importance.csv")

# # ==============================================================================
# # VISUALIZATION: Feature Importance
# # ==============================================================================
# fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# # Plot 1: Top individual features
# ax = axes[0]
# top_n = 25
# top_features = importance_df.head(top_n)
# y_pos = np.arange(len(top_features))
# colors = plt.cm.viridis(np.linspace(0, 1, top_n))

# ax.barh(y_pos, top_features['importance_mean'], 
#         xerr=top_features['importance_std'],
#         alpha=0.8, edgecolor='black', color=colors, linewidth=0.5)
# ax.set_yticks(y_pos)
# ax.set_yticklabels(top_features['feature'], fontsize=8)
# ax.set_xlabel('Mean Importance', fontsize=11)
# ax.set_title(f'Top {top_n} Most Important Features', fontsize=12, fontweight='bold')
# ax.invert_yaxis()
# ax.grid(True, alpha=0.3, axis='x')

# Plot 2: Group-level importance
ax = axes[1]
groups = list(group_importance.index)
x_pos = np.arange(len(groups))
colors_group = plt.cm.Set3(np.linspace(0, 1, len(groups)))
bars = ax.bar(x_pos, group_importance.values, 
              color=colors_group, edgecolor='black', linewidth=1.5, alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(groups, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('Total Importance', fontsize=11)
ax.set_title('Feature Importance by Group', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, val in zip(bars, group_importance.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('spatiotemporal_feature_importance.png', dpi=300, bbox_inches='tight')
print(f"✓ Feature importance plot saved to: spatiotemporal_feature_importance.png")
plt.show()

# ==============================================================================
# KEY INSIGHTS
# ==============================================================================
print(f"\n" + "="*80)
print("FEATURE IMPORTANCE INSIGHTS")
print("="*80)

top_group = group_importance.index[0]
top_group_val = group_importance.values[0]
print(f"\nMost important feature group: {top_group}")
print(f"  Total importance: {top_group_val:.4f}")
print(f"  Number of features: {(importance_df['group'] == top_group).sum()}")

print(f"\nTop 5 individual features:")
for i, row in importance_df.head(5).iterrows():
    print(f"  {i+1}. {row['feature']} (group: {row['group']}, importance: {row['importance_mean']:.4f})")

In [ ]:

# df = importance_df.copy()
# group_mapping = {
#     'overridding_plate_properties': 'Overriding Plate',
#     'incoming_plate_properties': 'Incoming Plate',
#     'slab_geometry': 'Slab Geometry',
#     'plate_kinematics': 'Plate Kinematics',
#     'surface_forcing': 'Surface Forcing',
#     'structural_heterogeneities': 'Structural Heterogeneities'
# }

# df['group_clean'] = (
#     df['group']
#     .map(group_mapping)
# )



# # 2. Intuitive Cleaning for Spatiotemporal Labels
# def clean_spatiotemporal_name(name):
#     replacements = {
#         'crustal_thickness_mean (m)': 'Mean Crustal Thickness',
#         'crustal_thickness_median (m)': 'Median Crustal Thickness',
#         'crustal_thickness_min (m)': 'Minimum Crustal Thickness',
#         'total_precipitation (km)': 'Paleo-Precipitation',
#         'distance_to_trench (km)': 'Arc-Trench Distance',
#         'seafloor_age (Ma)': 'Subducting Slab Age',
#         'seamount_distance': 'Proximity to Seamounts',
#         'convergence_rate_orthogonal (cm/yr)': 'Orthogonal Convergence Speed',
#         'trench_velocity (cm/yr)': 'Trench Migration Velocity',
#         'seafloor_spreading_rate (km/Myr)': 'Ridge Spreading Rate',
#         'convergence_obliquity (degrees)': 'Convergence Angle (Obliquity)',
#         'trench_velocity_parallel (cm/yr)': 'Strike-Parallel Trench Motion',
#         'subducted_plate_volume (m)': 'Total Subducted Slab Volume',
#         'fz_magnitude': 'Fracture Zone Magnitude',
#         'carbonate_thickness (m)': 'Sedimentary Carbonate Thickness',
#         'subducted_water_volume (m)': 'Subducted Pore Water Volume',
#         'slab_flux (m^2/yr)': 'Total Slab Flux',
#         'subducted_carbonates_volume (m)': 'Subducted Carbonate Volume',
#         'LIP_distance': 'Proximity to Conjugate LIP',
#     }
#     return replacements.get(name, name)

# df['clean_feature'] = df['feature'].apply(clean_spatiotemporal_name)

# # 3. Plotting
# plt.figure(figsize=(12, 8), dpi=300)
# sns.set_style("whitegrid", {'axes.grid': False})

# # Geodynamic Palette
# palette = {
#     'Overriding Plate': '#4e79a7',
#     'Surface Forcing': '#e15759',
#     'Slab Geometry': '#76b7b2',
#     'Plate Kinematics': '#f28e2b',
#     'Structural Heterogeneities': '#59a14f',
#     'Incoming Plate': '#edc948'
# }

# ax = sns.barplot(
#     x='importance_mean', 
#     y='clean_feature', 
#     hue='group_clean', 
#     data=df[:5], 
#     dodge=False, 
#     palette=palette,
#     edgecolor='0.2'
# )

# # 4. Final Polish
# plt.title('Top Spatiotemporal Drivers of Porphyry Systems', fontsize=18, fontweight='bold', loc='left', pad=25)
# plt.xlabel('Relative Importance (Mean Gain)', fontsize=14)
# plt.ylabel('')

# # Log scale is often useful when one feature (Crustal Thickness) dominates the rest
# # plt.xscale('log') # Uncomment if you want to see the smaller bars more clearly

# plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)

# # Annotate the dominant feature
# # plt.text(0.57, 0, 'Dominant Driver', color='#4e79a7', fontweight='bold', va='center')

# sns.despine()
# plt.tight_layout()
# plt.show()

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# # ---------------------------
# # 1. Prepare Top-5 features
# # ---------------------------
# top5 = (
#     df.sort_values('importance_mean', ascending=False)
#       .head(5)
# )

# # 
# # ---------------------------
# # 1. Define limits
# # ---------------------------
# x_left_max = 0.08     # zoomed-in small features
# x_right_min = 0.45    # start of dominant feature
# x_right_max = 0.5     # a little above the dominant feature

# # ---------------------------
# # 2. Create figure and axes
# # ---------------------------
# fig, (ax1, ax2) = plt.subplots(
#     1, 2, sharey=True, figsize=(14, 6),
#     gridspec_kw={'width_ratios': [3, 1]}
# )

# # LEFT: zoomed-in small features
# sns.barplot(
#     x='importance_mean',
#     y='clean_feature',
#     hue='group_clean',
#     data=top5,
#     dodge=False,
#     palette=palette,
#     edgecolor='0.2',
#     ax=ax1
# )
# ax1.set_xlim(0, x_left_max)
# ax1.set_xlabel('Relative Importance (Mean Gain)')
# ax1.set_ylabel('')
# ax1.legend_.remove()

# # RIGHT: dominant feature
# sns.barplot(
#     x='importance_mean',
#     y='clean_feature',
#     hue='group_clean',
#     data=top5,
#     dodge=False,
#     palette=palette,
#     edgecolor='0.2',
#     ax=ax2
# )
# ax2.set_xlim(x_right_min, x_right_max)
# ax2.set_xlabel('')
# ax2.set_ylabel('')
# ax2.legend_.remove()

# # ---------------------------
# # 3. Broken-axis marks
# # ---------------------------
# d = 0.015  # size in axes fraction
# kwargs = dict(color='k', lw=1.5, clip_on=False)

# # vertical zig-zags at the edges
# ax1.plot((1-d, 1+d), (-d, +d), transform=ax1.transAxes, **kwargs)
# ax1.plot((1-d, 1+d), (1-d, 1+d), transform=ax1.transAxes, **kwargs)

# ax2.plot((-d, +d), (-d, +d), transform=ax2.transAxes, **kwargs)
# ax2.plot((-d, +d), (1-d, 1+d), transform=ax2.transAxes, **kwargs)

# # ---------------------------
# # 4. Title & polish
# # ---------------------------
# fig.suptitle(
#     'Top Spatiotemporal Drivers of Porphyry Systems',
#     fontsize=18,
#     fontweight='bold',
#     x=0.1,
#     ha='left'
# )

# sns.despine()
# plt.tight_layout()
# plt.show()


In [ ]:
# ==============================================================================
# STEP 8.5: WEIGHTED vs UNWEIGHTED MODEL COMPARISON
# ==============================================================================
print("\n" + "="*80)
print("STEP 8.5: Compare Feature Importance WITH and WITHOUT Tonnage Weighting")
print("="*80)

print("\nThis analysis reveals:")
print("  1. Which features are emphasized by large deposits (tonnage-weighted model)")
print("  2. Which features are universal predictors (unweighted model)")
print("  3. Genetic differences in feature patterns between large and small deposits")

# Train UNWEIGHTED model for comparison
print("\n" + "-"*80)
print("Training UNWEIGHTED model (no tonnage influence)...")
print("-"*80)

# Use same base estimator configuration
base_estimator_unweighted = xgb.XGBClassifier(
    max_depth=6,
    learning_rate=0.1,
    n_estimators=100,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    eval_metric='logloss',
    use_label_encoder=False
)

pu_model_unweighted = PUBaggingClassifier(
    base_estimator=base_estimator_unweighted,
    n_bags=10,
    bag_size=0.8,
    positive_label=1,
    unlabeled_label=0,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)

# Train WITHOUT sample weights (uniform weights)
print("Training with uniform weights (all deposits equal)...")
uniform_weights = np.ones(len(y_train))
pu_model_unweighted.fit(X_train, y_train, sample_weight=uniform_weights)
print("✓ Unweighted model training complete")

# Get feature importances from both models
importance_weighted_mean, importance_weighted_std = pu_model.get_feature_importances()
importance_unweighted_mean, importance_unweighted_std = pu_model_unweighted.get_feature_importances()

# Create comparison DataFrame
importance_comparison = pd.DataFrame({
    'feature': X_train.columns,
    'importance_weighted': importance_weighted_mean,
    'importance_unweighted': importance_unweighted_mean,
    'difference': importance_weighted_mean - importance_unweighted_mean,
    'ratio': importance_weighted_mean / (importance_unweighted_mean + 1e-10),
})

# Assign geological groups
importance_comparison['group'] = importance_comparison['feature'].map(group_mapping)

# Sort by absolute difference
importance_comparison['abs_difference'] = np.abs(importance_comparison['difference'])
importance_comparison_sorted = importance_comparison.sort_values('abs_difference', ascending=False)

print(f"\n{'='*80}")
print("FEATURE IMPORTANCE COMPARISON RESULTS")
print(f"{'='*80}")

print("\nTop 15 features EMPHASIZED by tonnage weighting (large deposits):")
print("(Positive difference = more important in weighted model)")
top_weighted = importance_comparison_sorted[importance_comparison_sorted['difference'] > 0].head(15)
if len(top_weighted) > 0:
    print(top_weighted[['feature', 'importance_weighted', 'importance_unweighted', 'difference', 'group']].to_string(index=False))
else:
    print("  No features emphasized by weighting")

print("\n\nTop 15 features DE-EMPHASIZED by tonnage weighting:")
print("(Negative difference = more important in unweighted model)")
top_unweighted = importance_comparison_sorted[importance_comparison_sorted['difference'] < 0].head(15)
if len(top_unweighted) > 0:
    print(top_unweighted[['feature', 'importance_weighted', 'importance_unweighted', 'difference', 'group']].to_string(index=False))
else:
    print("  No features de-emphasized by weighting")

# Group-level comparison
print("\n\nGeological Group-Level Comparison:")
group_comparison = importance_comparison.groupby('group').agg({
    'importance_weighted': 'sum',
    'importance_unweighted': 'sum',
    'difference': 'sum'
}).sort_values('difference', ascending=False)
print(group_comparison.to_string())

# Evaluate both models on test set
y_pred_unweighted, _ = pu_model_unweighted.predict_with_uncertainty(X_test)
auc_roc_unweighted = roc_auc_score(y_test, y_pred_unweighted)
auc_pr_unweighted = average_precision_score(y_test, y_pred_unweighted)

print(f"\n{'='*80}")
print("MODEL PERFORMANCE COMPARISON:")
print(f"{'='*80}")
print(f"Weighted Model (Tonnage-aware):")
print(f"  AUC-ROC: {auc_roc:.4f}")
print(f"  AUC-PR:  {auc_pr:.4f}")
print(f"\nUnweighted Model (Uniform):")
print(f"  AUC-ROC: {auc_roc_unweighted:.4f}")
print(f"  AUC-PR:  {auc_pr_unweighted:.4f}")

# ==============================================================================
# VISUALIZATION: Weighted vs Unweighted Comparison
# ==============================================================================
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Plot 1: Scatter - Weighted vs Unweighted importance
ax = axes[0, 0]
scatter = ax.scatter(importance_comparison['importance_unweighted'], 
                    importance_comparison['importance_weighted'],
                    c=importance_comparison['abs_difference'], 
                    cmap='coolwarm', s=30, alpha=0.6, edgecolors='black', linewidth=0.3)

# Add diagonal line (y=x)
max_val = max(importance_comparison['importance_weighted'].max(), 
              importance_comparison['importance_unweighted'].max())
ax.plot([0, max_val], [0, max_val], 'k--', linewidth=2, alpha=0.5, label='Equal Importance')

# Annotate top different features
top_diff_features = importance_comparison_sorted.head(10)
for _, row in top_diff_features.iterrows():
    ax.annotate(row['feature'][:20], 
               (row['importance_unweighted'], row['importance_weighted']),
               fontsize=7, alpha=0.7)

ax.set_xlabel('Importance (Unweighted Model)', fontsize=11)
ax.set_ylabel('Importance (Weighted Model)', fontsize=11)
ax.set_title('Feature Importance: Weighted vs Unweighted', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax, label='Absolute Difference')

# Plot 2: Top features emphasized by tonnage
ax = axes[0, 1]
top_n_comp = 20
if len(top_weighted) > 0:
    top_emphasized = importance_comparison_sorted[importance_comparison_sorted['difference'] > 0].head(top_n_comp)
    if len(top_emphasized) > 0:
        y_pos = np.arange(len(top_emphasized))
        bars = ax.barh(y_pos, top_emphasized['difference'], color='coral', 
                      edgecolor='black', linewidth=0.5, alpha=0.8)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(top_emphasized['feature'], fontsize=7)
        ax.set_xlabel('Importance Increase (Weighted - Unweighted)', fontsize=11)
        ax.set_title(f'Top {len(top_emphasized)} Features Emphasized by Large Deposits', fontsize=12, fontweight='bold')
        ax.invert_yaxis()
        ax.grid(True, alpha=0.3, axis='x')
    else:
        ax.text(0.5, 0.5, 'No features emphasized\nby tonnage weighting', 
               ha='center', va='center', transform=ax.transAxes, fontsize=12)
        ax.axis('off')
else:
    ax.text(0.5, 0.5, 'No features emphasized\nby tonnage weighting', 
           ha='center', va='center', transform=ax.transAxes, fontsize=12)
    ax.axis('off')

# Plot 3: Top features de-emphasized by tonnage
ax = axes[1, 0]
if len(top_unweighted) > 0:
    top_deemphasized = importance_comparison_sorted[importance_comparison_sorted['difference'] < 0].head(top_n_comp)
    if len(top_deemphasized) > 0:
        y_pos = np.arange(len(top_deemphasized))
        bars = ax.barh(y_pos, np.abs(top_deemphasized['difference']), color='skyblue', 
                      edgecolor='black', linewidth=0.5, alpha=0.8)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(top_deemphasized['feature'], fontsize=7)
        ax.set_xlabel('Importance Decrease (|Weighted - Unweighted|)', fontsize=11)
        ax.set_title(f'Top {len(top_deemphasized)} Features More Important for Small Deposits', fontsize=12, fontweight='bold')
        ax.invert_yaxis()
        ax.grid(True, alpha=0.3, axis='x')
    else:
        ax.text(0.5, 0.5, 'No features de-emphasized\nby tonnage weighting', 
               ha='center', va='center', transform=ax.transAxes, fontsize=12)
        ax.axis('off')
else:
    ax.text(0.5, 0.5, 'No features de-emphasized\nby tonnage weighting', 
           ha='center', va='center', transform=ax.transAxes, fontsize=12)
    ax.axis('off')

# Plot 4: Group-level comparison
ax = axes[1, 1]
groups = list(group_comparison.index)
x_pos = np.arange(len(groups))
width = 0.35

bars1 = ax.bar(x_pos - width/2, group_comparison['importance_weighted'], width, 
              label='Weighted (Tonnage)', color='coral', edgecolor='black', linewidth=1, alpha=0.8)
bars2 = ax.bar(x_pos + width/2, group_comparison['importance_unweighted'], width, 
              label='Unweighted (Uniform)', color='skyblue', edgecolor='black', linewidth=1, alpha=0.8)

ax.set_xticks(x_pos)
ax.set_xticklabels(groups, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('Total Importance', fontsize=11)
ax.set_title('Geological Group Importance: Weighted vs Unweighted', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Add difference annotations
for i, (g, diff) in enumerate(zip(groups, group_comparison['difference'])):
    if diff > 0.001:  # Only show significant differences
        ax.text(i, group_comparison.loc[g, 'importance_weighted'] + 0.01, 
               f'+{diff:.3f}', ha='center', va='bottom', fontsize=8, color='red', fontweight='bold')
    elif diff < -0.001:
        ax.text(i, group_comparison.loc[g, 'importance_unweighted'] + 0.01, 
               f'{diff:.3f}', ha='center', va='bottom', fontsize=8, color='blue', fontweight='bold')

plt.tight_layout()
plt.savefig('spatiotemporal_weighted_vs_unweighted_comparison.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Comparison plots saved to: spatiotemporal_weighted_vs_unweighted_comparison.png")
plt.show()

# Save comparison results
importance_comparison_sorted.to_csv('spatiotemporal_importance_comparison.csv', index=False)
print(f"✓ Feature importance comparison saved to: spatiotemporal_importance_comparison.csv")

group_comparison.to_csv('spatiotemporal_group_comparison.csv')
print(f"✓ Group importance comparison saved to: spatiotemporal_group_comparison.csv")

# Interpretation
print(f"\n{'='*80}")
print("INTERPRETATION & GEOLOGICAL INSIGHTS:")
print(f"{'='*80}")

if len(group_comparison) > 0:
    most_emphasized_group = group_comparison['difference'].idxmax()
    most_deemphasized_group = group_comparison['difference'].idxmin()
    
    if group_comparison['difference'].max() > 0.001:
        print(f"\nMost emphasized by large deposits: {most_emphasized_group}")
        print(f"  → Large deposits show stronger signals in {most_emphasized_group} features")
        print(f"  → These may represent more intense mineralization signatures")
    
    if group_comparison['difference'].min() < -0.001:
        print(f"\nMost emphasized by small deposits: {most_deemphasized_group}")
        print(f"  → Small deposits rely more on {most_deemphasized_group} features")
        print(f"  → These may be more subtle or specific indicators")

print(f"\nKey findings:")
print(f"  1. Features with positive difference → more predictive of LARGE deposits")
print(f"  2. Features with negative difference → more predictive of SMALL deposits")
print(f"  3. Features near diagonal → universal predictors (all sizes)")

print(f"\nRecommendations:")
print(f"  - Use WEIGHTED model for targeting large economic deposits")
print(f"  - Use UNWEIGHTED model for comprehensive deposit discovery")
print(f"  - Combine both models for balanced exploration strategy")

print("\n" + "="*80)

---

## 🔬 Weighted vs Unweighted Model Comparison

### Why Compare?

Tonnage weighting can introduce **bias** toward features that characterize large deposits. By comparing weighted and unweighted models, we can:

1. **Identify size-dependent features** - Which features are more predictive of large vs small deposits
2. **Understand genetic differences** - Different mineralization processes may produce different deposit sizes
3. **Optimize exploration strategy** - Choose the right model based on target size
4. **Validate weighting impact** - Ensure tonnage weighting improves rather than distorts predictions

### What This Analysis Shows

- **Features with POSITIVE difference** → More important in weighted model → Predictive of LARGE deposits
- **Features with NEGATIVE difference** → More important in unweighted model → Predictive of SMALL deposits
- **Features near diagonal** → Similar importance in both models → Universal predictors (all sizes)

### Practical Applications

| Target | Recommended Model | Reasoning |
|--------|------------------|-----------|
| **Major economic deposits (>10 Mt)** | Weighted (tonnage-aware) | Optimized for large deposit signatures |
| **Comprehensive discovery** | Unweighted (uniform) | Treats all deposits equally |
| **Balanced exploration** | Ensemble of both | Captures both large and small deposits |

---

In [ ]:
# ==============================================================================
# STEP 9: IDENTIFY HIGH-PROSPECTIVITY UNLABELED LOCATIONS
# ==============================================================================
print("\n" + "="*80)
print("STEP 9: High-Prospectivity Unlabeled Analysis")
print("="*80)

print("\nIn PU learning, high-scoring unlabeled samples may represent:")
print("  1. Undiscovered mineral deposits")
print("  2. Areas with similar characteristics to known deposits")
print("  3. Priority targets for future exploration")

# Get unlabeled samples and their predictions
unlabeled_mask = y_test == 0
n_unlabeled = np.sum(unlabeled_mask)
unlabeled_predictions = y_pred_mean[unlabeled_mask]

# Define high-prospectivity thresholds based on deposit distribution
deposit_prediction_threshold_90 = np.percentile(deposit_predictions, 10)  # Lower 10% of deposits
deposit_prediction_threshold_50 = np.percentile(deposit_predictions, 50)  # Median deposit score
deposit_prediction_threshold_25 = np.percentile(deposit_predictions, 75)  # Upper 25% of deposits

print(f"\nKnown deposit prediction score benchmarks:")
print(f"  10th percentile (lower deposits): {deposit_prediction_threshold_90:.3f}")
print(f"  50th percentile (median deposit): {deposit_prediction_threshold_50:.3f}")
print(f"  75th percentile (high-quality deposits): {deposit_prediction_threshold_25:.3f}")

# Count unlabeled samples exceeding these thresholds
unlabeled_above_90 = np.sum(unlabeled_predictions >= deposit_prediction_threshold_90)
unlabeled_above_50 = np.sum(unlabeled_predictions >= deposit_prediction_threshold_50)
unlabeled_above_25 = np.sum(unlabeled_predictions >= deposit_prediction_threshold_25)

print(f"\nUnlabeled samples with deposit-like characteristics:")
print(f"  Above 10th percentile of deposits: {unlabeled_above_90:,} ({unlabeled_above_90/n_unlabeled*100:.2f}%)")
print(f"  Above median deposit score: {unlabeled_above_50:,} ({unlabeled_above_50/n_unlabeled*100:.2f}%)")
print(f"  Above 75th percentile of deposits: {unlabeled_above_25:,} ({unlabeled_above_25/n_unlabeled*100:.2f}%)")

# High-prospectivity threshold
high_prosp_threshold = deposit_prediction_threshold_50
high_prosp_unlabeled_mask = unlabeled_mask & (y_pred_mean >= high_prosp_threshold)
n_high_prosp = np.sum(high_prosp_unlabeled_mask)

print(f"\n" + "="*80)
print(f"EXPLORATION RECOMMENDATION")
print("="*80)
print(f"Identified {n_high_prosp:,} high-prospectivity unlabeled locations")
print(f"(prediction score ≥ {high_prosp_threshold:.3f}, matching median known deposit)")
print("="*80)

# Get indices and save for exploration
if hasattr(X_test, 'index'):
    high_prosp_indices = X_test.index[high_prosp_unlabeled_mask]
else:
    high_prosp_indices = np.where(high_prosp_unlabeled_mask)[0]

high_prosp_df = pd.DataFrame({
    'test_index': high_prosp_indices,
    'prediction_score': y_pred_mean[high_prosp_unlabeled_mask],
    'prediction_uncertainty': y_pred_std[high_prosp_unlabeled_mask],
    'rank_percentile': [(np.where(all_ranked_indices == idx)[0][0] + 1) / n_total * 100 
                        for idx in np.where(high_prosp_unlabeled_mask)[0]]
})
high_prosp_df = high_prosp_df.sort_values('prediction_score', ascending=False)

# ==============================================================================
# VISUALIZATION: High-Prospectivity Analysis
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Histogram comparison
ax = axes[0]
bins = np.linspace(0, 1, 50)
ax.hist(deposit_predictions, bins=bins, alpha=0.7, label=f'Known Deposits (n={n_deposits})', 
        color='red', density=True, edgecolor='black', linewidth=0.5)
ax.hist(unlabeled_predictions, bins=bins, alpha=0.5, label=f'Unlabeled (n={n_unlabeled:,})', 
        color='blue', density=True, edgecolor='black', linewidth=0.5)

# Add threshold lines
ax.axvline(deposit_prediction_threshold_50, color='darkred', linestyle='--', linewidth=2, 
           label=f'Median Deposit ({deposit_prediction_threshold_50:.3f})')
ax.axvline(deposit_prediction_threshold_25, color='purple', linestyle='--', linewidth=2, 
           label=f'75th %ile Deposit ({deposit_prediction_threshold_25:.3f})')

ax.set_xlabel('Prediction Score', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Prediction Distribution: Known Deposits vs Unlabeled', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)

# Highlight overlap region
ax.axvspan(deposit_prediction_threshold_50, 1.0, alpha=0.2, color='yellow')
ax.text(0.7, ax.get_ylim()[1]*0.9, 
        f'{unlabeled_above_50:,} unlabeled\nabove median\ndeposit score', 
        fontsize=10, bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

# Plot 2: Cumulative distribution
ax = axes[1]
sorted_deposit_preds = np.sort(deposit_predictions)
sorted_unlabeled_preds = np.sort(unlabeled_predictions)

ax.plot(sorted_deposit_preds, np.linspace(0, 1, len(sorted_deposit_preds)), 
        'r-', linewidth=2.5, label='Known Deposits (CDF)')
ax.plot(sorted_unlabeled_preds, np.linspace(0, 1, len(sorted_unlabeled_preds)), 
        'b-', linewidth=2.5, alpha=0.7, label='Unlabeled (CDF)')

ax.axvline(deposit_prediction_threshold_50, color='darkred', linestyle='--', linewidth=2, alpha=0.7)
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, alpha=0.5)

ax.set_xlabel('Prediction Score', fontsize=12)
ax.set_ylabel('Cumulative Probability', fontsize=12)
ax.set_title('Cumulative Distribution: High-Prospectivity Unlabeled', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

ax.text(deposit_prediction_threshold_50 + 0.05, 0.25, 
        f'{unlabeled_above_50:,} unlabeled\nexceed median\ndeposit score', 
        fontsize=10, bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('spatiotemporal_potential_discoveries.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Potential discoveries analysis saved to: spatiotemporal_potential_discoveries.png")
plt.show()

# Save high-prospectivity targets
high_prosp_df.to_csv('spatiotemporal_high_prospectivity_targets.csv', index=False)
print(f"✓ High-prospectivity unlabeled targets saved to: spatiotemporal_high_prospectivity_targets.csv")
print(f"\nTop 10 highest-scoring unlabeled targets:")
print(high_prosp_df.head(10).to_string(index=False))

print("\n" + "="*80)
print("SPATIOTEMPORAL PU LEARNING WORKFLOW COMPLETE")
print("="*80)
print(f"\nSummary:")
print(f"  Training samples: {len(y_train):,}")
print(f"  Test samples: {len(y_test):,}")
print(f"  Known deposits in test: {n_deposits}")
print(f"  Top 5% recall: {top5_metrics['Recall@K']*100:.1f}%")
print(f"  Top 5% enrichment: {top5_metrics['Enrichment']:.1f}x")
print(f"  High-prospectivity targets identified: {n_high_prosp:,}")
print("="*80)

## Step 10: Predict on New Grid Data

This section loads external grid data and makes predictions using the trained PU model.

**Key Steps:**
1. Load the grid data from CSV
2. Apply the same preprocessing as training data:
   - Handle missing values
   - Remove zero-variance features
   - Drop correlated features (if correlation analysis was used)
   - Ensure feature alignment with training features
3. Make predictions with uncertainty quantification
4. Save predictions to CSV
5. Visualize prediction distribution

**Output Files:**
- `spatiotemporal_grid_predictions.csv`: Full prediction results with coordinates and metadata
- `spatiotemporal_grid_prediction_distribution.png`: Distribution plot comparing grid predictions with training data

In [ ]:
# grid_data_path = "<HOME>/Downloads/ApplicationKalpa/prepared_data/grid_data.csv"
# print(f"\nLoading grid data from: {grid_data_path}")
# grid_df = pd.read_csv(grid_data_path)
# grid_df.columns

In [ ]:
# pd.read_csv("<DATA_ROOT>/CopperLithium/NorthAmerica/grid/grid_data_170to0.csv").columns

In [ ]:
print("=" * 80)
print("STEP 10: PREDICT ON NEW GRID DATA")
print("=" * 80)

# Load the grid data
grid_data_path = "<DATA_ROOT>/Paper/Zenodo_DataBundle/grid_data/grid_data_170to0.csv"
print(f"\nLoading grid data from: {grid_data_path}")

try:
    grid_df = pd.read_csv(grid_data_path)
    print(f"✓ Loaded {len(grid_df):,} grid points")
    print(f"  Columns: {grid_df.shape[1]}")
    
    # Identify metadata columns (assume similar structure to training data)
    # Common metadata: lat, lon, time, x, y, grid_id, cell_id, etc.
    potential_metadata = ['lat', 'lon', 'latitude', 'longitude', 'time', 'age (Ma)', 'present_lat', 'present_lon', 
                         'x', 'y', 'grid_id', 'cell_id', 'point_id', 'id']
    grid_metadata_cols = [col for col in grid_df.columns if col.lower() in potential_metadata]
    
    # Identify feature columns (everything that's not metadata)
    grid_feature_cols = [col for col in grid_df.columns if col not in grid_metadata_cols]
    
    print(f"\n  Metadata columns: {grid_metadata_cols}")
    print(f"  Feature columns: {len(grid_feature_cols)}")
    
    # Extract features
    X_grid = grid_df[grid_feature_cols].copy()
    print(f"\n✓ Extracted features: {X_grid.shape}")
    
except FileNotFoundError:
    print(f"✗ Error: File not found at {grid_data_path}")
    print("  Please check the file path and try again.")
    raise
except Exception as e:
    print(f"✗ Error loading grid data: {e}")
    raise

print("\n" + "-" * 80)
print("PREPROCESSING GRID DATA (matching training preprocessing)")
print("-" * 80)


In [ ]:
# Compare column names directly
train_cols = set(X_train.columns)
grid_cols = set(X_grid.columns)

print("--- Column Name Analysis ---")
print(f"Features only in Training: {train_cols - grid_cols}")
print(f"Features only in Grid: {grid_cols - train_cols}")

# Check for similar names (case sensitivity or typos)
import difflib
for missing in (train_cols - grid_cols):
    suggestions = difflib.get_close_matches(missing, list(grid_cols))
    if suggestions:
        print(f"Found match for '{missing}': maybe you meant {suggestions}?")

In [ ]:
# Show which layers have the most holes
missing_counts = X_grid.isnull().sum()
print(missing_counts[missing_counts > 0].sort_values(ascending=False))

In [ ]:

# Step 1: Handle missing values (same as training)
print("\n1. Handling missing values...")
n_missing_before = X_grid.isnull().sum().sum()
X_grid = X_grid.fillna(X_grid.median())
n_missing_after = X_grid.isnull().sum().sum()
print(f"   Missing values: {n_missing_before:,} → {n_missing_after:,}")

# Step 2: Remove zero-variance features (same as training)
print("\n2. Removing zero-variance features...")
variance = X_grid.var()
zero_var_cols_grid = variance[variance == 0].index.tolist()
if zero_var_cols_grid:
    print(f"   Found {len(zero_var_cols_grid)} zero-variance features")
    X_grid = X_grid.drop(columns=zero_var_cols_grid)
else:
    print(f"   No zero-variance features found")

# Step 3: Drop correlated features if they were dropped during training
if 'dropped_correlated_features' in locals() or 'dropped_correlated_features' in globals():
    print("\n3. Dropping correlated features (from training)...")
    correlated_to_drop = [col for col in dropped_correlated_features if col in X_grid.columns]
    if correlated_to_drop:
        X_grid = X_grid.drop(columns=correlated_to_drop)
        print(f"   Dropped {len(correlated_to_drop)} correlated features")
    else:
        print(f"   No correlated features to drop in grid data")
else:
    print("\n3. No correlation filtering applied during training")

# Step 4: Align features with training data
print("\n4. Aligning features with training data...")
training_features = X_train.columns.tolist()
grid_features = X_grid.columns.tolist()

# Features in grid but not in training
extra_features = [col for col in grid_features if col not in training_features]
if extra_features:
    print(f"   Dropping {len(extra_features)} features not in training data:")
    print(f"   {extra_features[:5]}{'...' if len(extra_features) > 5 else ''}")
    X_grid = X_grid.drop(columns=extra_features)

# Features in training but not in grid
missing_features = [col for col in training_features if col not in X_grid.columns]
if missing_features:
    print(f"   Adding {len(missing_features)} missing features (filled with 0):")
    print(f"   {missing_features[:5]}{'...' if len(missing_features) > 5 else ''}")
    for col in missing_features:
        X_grid[col] = 0

# Reorder columns to match training data
X_grid = X_grid[training_features]
print(f"\n✓ Final grid features: {X_grid.shape}")
print(f"  Feature alignment: {'MATCHED' if list(X_grid.columns) == training_features else 'ERROR'}")

print("\n" + "-" * 80)
print("MAKING PREDICTIONS")
print("-" * 80)

# Make predictions
print("\nPredicting prospectivity scores...")
y_grid_pred_all = pu_model.predict_proba(X_grid)


In [ ]:
# Make predictions with uncertainty quantification
y_grid_pred_mean, y_grid_pred_std = pu_model.predict_with_uncertainty(X_grid)

# Add predictions to dataframe
grid_df['prospectivity_score'] = y_grid_pred_mean
grid_df['prospectivity_uncertainty'] = y_grid_pred_std
# Add predictions to dataframe
grid_df['Prospectivity Score'] = y_grid_pred_mean

print(f"\n✓ Predictions complete")
print(f"  Mean prospectivity score: {y_grid_pred_mean.mean():.3f}")
print(f"  Std prospectivity score: {y_grid_pred_mean.std():.3f}")
print(f"  Mean uncertainty: {y_grid_pred_std.mean():.3f}")
print(f"  Max uncertainty: {y_grid_pred_std.max():.3f}")

print(f"\nPrediction distribution:")
print(f"  Min: {y_grid_pred_mean.min():.4f}")
print(f"  25th percentile: {np.percentile(y_grid_pred_mean, 25):.4f}")
print(f"  Median: {np.percentile(y_grid_pred_mean, 50):.4f}")
print(f"  75th percentile: {np.percentile(y_grid_pred_mean, 75):.4f}")
print(f"  Max: {y_grid_pred_mean.max():.4f}")

In [ ]:
# Save predictions
output_path = 'spatiotemporal_grid_predictions_latest.csv'
grid_df.to_csv(output_path, index=False)
print(f"\n✓ Grid predictions saved to: {output_path}")

In [ ]:
# import pandas as pd
# import numpy as np

# n = 3
# threshold = 0.5

# # Make sure the df is sorted correctly
# df = grid_df.sort_values(['present_lat', 'present_lon', 'age (Ma)'], ascending=[True, True, False]).copy()

# def filter_consecutive(group):
#     # Convert to boolean: True if above threshold
#     above = group > threshold
    
#     # rolling window of size n, min_periods=n, compute sum
#     # If sum == n, then all in window are above threshold
#     consecutive_mask = above.rolling(n, min_periods=n).sum() == n
    
#     # Shift the mask backward to mark all n elements in the window
#     # Fill NaN with False
#     consecutive_mask = consecutive_mask.fillna(False)
#     for i in range(1, n):
#         consecutive_mask |= consecutive_mask.shift(-i, fill_value=False)
    
#     # Multiply original scores by mask
#     return group * consecutive_mask.astype(float)

# # Apply per grid point
# df['prospectivity_filtered'] = df.groupby(['present_lat','present_lon'])['Prospectivity Score'].transform(filter_consecutive)

# # Quick check
# print(df[['present_lat','present_lon','age (Ma)','Prospectivity Score','prospectivity_filtered']])

In [ ]:
# from numba import jit

# n = 3
# threshold = 0.5


# @jit(nopython=True)
# def mark_consecutive(values, threshold, n):
#     result = np.zeros_like(values)
#     length = len(values)
    
#     for i in range(length - n + 1):
#         # Check if next n values are all above threshold
#         all_above = True
#         for j in range(n):
#             if values[i + j] <= threshold:
#                 all_above = False
#                 break
        
#         if all_above:
#             for j in range(n):
#                 result[i + j] = values[i + j]
    
#     return result

# # Apply
# df['prospectivity_filtered'] = df.groupby(['present_lat','present_lon'], group_keys=False)['Prospectivity Score'].apply(
#     lambda x: pd.Series(mark_consecutive(x.values, threshold, n), index=x.index)
# )

In [ ]:
# nc=df_to_NetCDF(df['present_lon'], df['present_lat'], df['prospectivity_filtered'], statistic='max',grid_resolution=0.25)
# nc.plot( cmap='RdYlGn', vmin=0, vmax=1)

In [ ]:
from pyDTDM.utils import *

In [ ]:
grid_resolution=0.25
# Compute statistics on the grid
probs_counts = df_to_NetCDF(
   grid_df['present_lon'], grid_df['present_lat'], grid_df['Prospectivity Score'],
    statistic='count',
    grid_resolution=grid_resolution
)
probs_counts.name = "counts"

probs_mean = df_to_NetCDF(
   grid_df['present_lon'], grid_df['present_lat'], grid_df['Prospectivity Score'],
    statistic='mean',
    grid_resolution=grid_resolution
)
probs_mean.name = "mean"

probs_max = df_to_NetCDF(
   grid_df['present_lon'], grid_df['present_lat'], grid_df['Prospectivity Score'],
    statistic='max',
    grid_resolution=grid_resolution
)
probs_max.name = "max"

probs_median = df_to_NetCDF(
   grid_df['present_lon'], grid_df['present_lat'], grid_df['Prospectivity Score'],
    statistic='median',
    grid_resolution=grid_resolution
)
probs_median.name = "median"

probs_std = df_to_NetCDF(
   grid_df['present_lon'], grid_df['present_lat'], grid_df['Prospectivity Score'],
    statistic='std',
    grid_resolution=grid_resolution
)
probs_std.name = "std"



# probs_ero = df_to_NetCDF(
#     combined_probs_df['Present Longitude'],
#     combined_probs_df['Present Latitude'],
#     combined_probs_df['Erosion (m)'],
#     statistic='mean',
#     grid_resolution=0.5
# )
# probs_ero.name = "Erosion (m)"

# Combine all statistics into a single DataFrame
counts_df = probs_counts.to_dataframe().reset_index()
counts_df['mean'] = probs_mean.to_dataframe().reset_index()['mean']
counts_df['max'] = probs_max.to_dataframe().reset_index()['max']
counts_df['median'] = probs_median.to_dataframe().reset_index()['median']
counts_df['std'] = probs_std.to_dataframe().reset_index()['std']
# counts_df['erosion'] = probs_ero.to_dataframe().reset_index()['std']
# counts_df now contains: longitude, latitude, counts, mean, max, median, std
# counts_df=counts_df[counts_df['counts']>10]

In [ ]:
nc=df_to_NetCDF(counts_df['Longitude'], counts_df['Latitude'], counts_df['max'], statistic='max',grid_resolution=0.25)

In [ ]:
import os



import pyproj
# Point pyproj to the correct PROJ data directory
os.environ["PROJ_LIB"] = "<CONDA>/envs/EBMTest311/share/proj"
pyproj.datadir.set_data_dir(os.environ["PROJ_LIB"])
coastlines_file="<DATA_ROOT>/Raw/plate_model/StaticGeometries/Coastlines/Global_coastlines_low_res.shp"
coastlines_gdf=gpd.read_file(coastlines_file)


In [ ]:
deposits=training_df[training_df['label_binary']==1].copy()
deposits['tonnage_for_plot']=deposits['tonnage_mt_clean'].fillna(1)
deposits


In [ ]:
import xarray as xr
import numpy as np
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt

# Note: Global style settings are applied at the start of the notebook
fig_scale = 0.8
# 2-row figure
fig, axes = plt.subplots(2, 1, dpi=300, figsize=(10*fig_scale, 16*fig_scale))


ax = axes[0]


marker_sizes = 25 * np.log10(deposits['tonnage_for_plot'] + 1)
# Gaussian smoothing (returns ndarray)
nc2_array = nan_gaussian_filter(nc, sigma=1.5,radius=1)

# wrap back to DataArray (preserve coords)
nc2 = xr.DataArray(
    nc2_array,
    coords=nc.coords,
    dims=nc.dims
)

# nc2 = xr.DataArray(
#     nc.values,
#     coords=nc.coords,
#     dims=nc.dims
# )
# create colormap and set NaN color
cmap = plt.get_cmap('YlOrRd').copy()
cmap.set_bad(color='white')  # or 'lightgrey'

# plot on axis and capture return object
im = nc2.plot(
    ax=ax,
    cmap=cmap,
    add_colorbar=True,
    vmin=0, vmax=1,
)

coastlines_gdf.plot(ax=ax, 
                    facecolor="lightgrey", 
                    edgecolor="grey", alpha=0.1)
    
# Overlay known deposits with single color, sized by tonnage
ax.scatter(deposits['present_lon'], deposits['present_lat'], 
            s=marker_sizes,
        #    c='cyan',  # Single high-contrast color
        facecolor="None",
        #    marker='*', 
            edgecolors='black', 
            linewidths=1.2, 
            alpha=0.3,
            zorder=5)
# colorbar label (xarray returns colorbar via mappable)
im.colorbar.set_label('Max Spatiotemporal Prospectivity Score')


# Create simplified legend for deposit sizes
size_legend_elements = [
    plt.scatter([], [], s=80 + 100*np.log10(1+1), 
                # marker='*', 
                # c='cyan', 
                facecolor='None',
                edgecolors='black', 
                linewidths=1.2, label='1 Mt'),
    plt.scatter([], [], s=80 + 100*np.log10(10+1), 
                # marker='*', 
                # c='cyan', 
                facecolor='None',
                edgecolors='black', linewidths=1.2, label='10 Mt'),
    plt.scatter([], [], s=80 + 100*np.log10(100+1), 
                # marker='*', 
                # c='cyan', 
                facecolor='None',
                edgecolors='black', linewidths=1.2, label='100 Mt'),
]
legend1 = ax.legend(handles=size_legend_elements, 
                    loc='upper right', fontsize=10, 
                    title=f'Known Deposits (n={len(deposits)})',
                    title_fontsize=11,
                    framealpha=0.95)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# ax.set_xlabel('')
ax.tick_params(axis='x', which='both', bottom=True, top=False, labelbottom=True)
# ax.set_title('Spatial Distribution (Smoothed)')

ax.set_xlim(-145, -100)
ax.set_ylim(25, 72)

for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1.2)

ax.set_frame_on(True)
ax.grid(False)
ax.set_aspect('equal', adjustable='box')
ax.set_axis_on()
ax.set_title('Spatiotemporal Prospectivity Map', 
            fontsize=13, fontweight='bold')


nc2.name="max_prospectivity"
grid_max=nc2.to_dataframe().reset_index().dropna()
# grid_max=grid_max[grid_max['max_prospectivity']>0.001]
# 2. High Prospectivity Areas (top 5%)
ax = axes[1]
threshold_95 = np.percentile(grid_max['max_prospectivity'], 95)
high_prosp = grid_max[grid_max['max_prospectivity'] >= threshold_95]

coastlines_gdf.plot(ax=ax, 
                    facecolor="lightgrey", 
                    edgecolor="grey", alpha=0.1)

ax.scatter(grid_max['Longitude'][::50],
           grid_max['Latitude'][::50],
           c='none', edgecolors='none')

ax.set_facecolor('white')
# High prospectivity areas
scatter = ax.scatter(high_prosp['Longitude'], high_prosp['Latitude'], 
                    c=high_prosp['max_prospectivity'], cmap='inferno_r', 
                    s=20, alpha=0.8,
                    linewidths=0.3, vmin=threshold_95, vmax=1)


# Overlay known deposits with single color, sized by tonnage
ax.scatter(deposits['present_lon'], deposits['present_lat'], 
            s=marker_sizes,
        #    c='cyan',  # Single high-contrast color
        facecolor="None",
        #    marker='*', 
            edgecolors='black', 
            linewidths=1.2, 
            alpha=0.05,
            zorder=5)



# Colorbar for prospectivity only
cbar_prosp = plt.colorbar(scatter, ax=ax, label='Max Spatiotemporal Prospectivity Score')
    
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

ax.set_xlim(-145, -100)
ax.set_ylim(25, 72)

ax.set_aspect('equal', adjustable='box')
ax.set_title(f'High Prospectivity Areas (Top 5%)', 
            fontsize=13, fontweight='bold')

for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1.2)

ax.set_frame_on(True)
ax.grid(False)
# --------------------------------------------------
plt.tight_layout()
plt.show()

In [ ]:
import xarray as xr
import numpy as np
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt

# Note: Global style settings are applied at the start of the notebook
fig_scale = 0.8
# 2-row figure
fig, axes = plt.subplots(2, 1, dpi=300, figsize=(10*fig_scale, 16*fig_scale))


ax = axes[0]


marker_sizes = 80 + 25 * np.log10(deposits['tonnage_for_plot'] + 1)
# Gaussian smoothing (returns ndarray)
nc2_array = nan_gaussian_filter(nc, sigma=1.5,radius=1)

# wrap back to DataArray (preserve coords)
nc2 = xr.DataArray(
    nc2_array,
    coords=nc.coords,
    dims=nc.dims
)
# create colormap and set NaN color
cmap = plt.get_cmap('YlOrRd').copy()
cmap.set_bad(color='white')  # or 'lightgrey'

# plot on axis and capture return object
im = nc2.plot(
    ax=ax,
    cmap=cmap,
    add_colorbar=True,
)

coastlines_gdf.plot(ax=ax, 
                    facecolor="lightgrey", 
                    edgecolor="grey", alpha=0.1)
    
# Overlay known deposits with single color, sized by tonnage
ax.scatter(deposits['present_lon'], deposits['present_lat'], 
            s=marker_sizes,
        #    c='cyan',  # Single high-contrast color
        facecolor="None",
        #    marker='*', 
            edgecolors='black', 
            linewidths=1.2, 
            alpha=0.3,
            zorder=5)
# colorbar label (xarray returns colorbar via mappable)
im.colorbar.set_label('Max Spatiotemporal Prospectivity Score')


# Create simplified legend for deposit sizes
size_legend_elements = [
    plt.scatter([], [], s=80 + 100*np.log10(1+1), 
                # marker='*', 
                # c='cyan', 
                facecolor='None',
                edgecolors='black', 
                linewidths=1.2, label='1 Mt'),
    plt.scatter([], [], s=80 + 100*np.log10(10+1), 
                # marker='*', 
                # c='cyan', 
                facecolor='None',
                edgecolors='black', linewidths=1.2, label='10 Mt'),
    plt.scatter([], [], s=80 + 100*np.log10(100+1), 
                # marker='*', 
                # c='cyan', 
                facecolor='None',
                edgecolors='black', linewidths=1.2, label='100 Mt'),
]
legend1 = ax.legend(handles=size_legend_elements, 
                    loc='upper right', fontsize=10, 
                    title=f'Known Deposits (n={len(deposits)})',
                    title_fontsize=11,
                    framealpha=0.95)
# ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

ax.set_xlabel('')
ax.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
# ax.set_title('Spatial Distribution (Smoothed)')

ax.set_xlim(-145, -100)
ax.set_ylim(25, 72)

for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1.2)

ax.set_frame_on(True)
ax.grid(False)
ax.set_aspect('equal', adjustable='box')
ax.set_axis_on()
ax.set_title('Spatiotemporal Prospectivity Map', 
            fontsize=13, fontweight='bold')


nc2.name="max_prospectivity"
grid_max=nc2.to_dataframe().reset_index().dropna()
grid_max=grid_max[grid_max['max_prospectivity']>0]
# 2. High Prospectivity Areas (top 5%)
ax = axes[1]
threshold_95 = np.percentile(grid_max['max_prospectivity'], 95)
high_prosp = grid_max[grid_max['max_prospectivity'] >= threshold_95]

coastlines_gdf.plot(ax=ax, 
                    facecolor="lightgrey", 
                    edgecolor="grey", alpha=0.1)

ax.scatter(grid_max['Longitude'][::50],
           grid_max['Latitude'][::50],
           c='none', edgecolors='none')

ax.set_facecolor('white')
# High prospectivity areas
scatter = ax.scatter(high_prosp['Longitude'], high_prosp['Latitude'], 
                    c=high_prosp['max_prospectivity'], cmap='inferno_r', 
                    s=20, alpha=0.8,
                    linewidths=0.3, vmin=threshold_95, vmax=1)


# Overlay known deposits with single color, sized by tonnage
ax.scatter(deposits['present_lon'], deposits['present_lat'], 
            s=marker_sizes,
        #    c='cyan',  # Single high-contrast color
        facecolor="None",
        #    marker='*', 
            edgecolors='black', 
            linewidths=1.2, 
            alpha=0.05,
            zorder=5)



# Colorbar for prospectivity only
cbar_prosp = plt.colorbar(scatter, ax=ax, label='Max Spatiotemporal Prospectivity Score')
    
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

ax.set_xlim(-145, -100)
ax.set_ylim(25, 72)

ax.set_aspect('equal', adjustable='box')
ax.set_title(f'High Prospectivity Areas (Top 5%)', 
            fontsize=13, fontweight='bold')

for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1.2)

ax.set_frame_on(True)
ax.grid(False)
# --------------------------------------------------
plt.tight_layout()
plt.show()

In [ ]:
nc2.to_netcdf("Max_STAMP.nc")

In [ ]:
create_directory_if_not_exists("<DATA_ROOT>/Raw/prospectivity_maps")
for t in sorted(grid_df['age (Ma)'].unique()):
    grid_dft=grid_df[grid_df['age (Ma)']==t].copy()
    nc=df_to_NetCDF(grid_dft['lon'], grid_dft['lat'], grid_dft['Prospectivity Score'], statistic='mean', grid_resolution=0.5,
                    lat_bin_edges=np.arange(-90-0.25, 90+0.25, 0.5),
                    lon_bin_edges=np.arange(-180-0.25, 180+0.25, 0.5))
    nc=nc.to_dataset(name='Prospectivity Score')
    nc.to_netcdf(f'<DATA_ROOT>/Raw/prospectivity_maps/prospectivity_map_{t:.1f}Ma.nc')

In [ ]:
nc.to_netcdf('spatiotemporal_grid_predictions_final.nc')